# **Notes**

**Flow Rate Profile Analysis**

Use with any microfluidic chip, place the black line over the channel to analyze flow across the channel. Specify number of channels and channel gaps.

***Feb 2022 - Lane Breshears***, modified from .py code Alex Day

***Mar 2023 - Yisha Tang***, modified for eNAMPT: check the example boards before running video analysis; merge multichannel data to one sheet and give threshold-cleaned flow profile

***Feb 2024 - Yisha Tang***, modified for PFAS: new section of data analysis and visualization to further clean the flow profile, analyze flow rate at adjustable time range.

***Mar 14- Yisha Tang***: Use the maxiuma variation to locate the initial start if variation doesn't meet the threshold; new funtion - drop the messy data.

***Apr 2024 - Yisha Tang***, goal for this modification:

Multiple line along one channel for flow extraction
ML framework (start from SVM)

***Jun 2024 - Jocelyn Reynolds*** Updated for the OneHealth Skin Microbiome project. The pipeline was modified to calculate flow velocity over 30-frame increments, with added visualization of the start-trim location on flow profile plots. Additional functionality was implemented to generate bacteria- and reagent-level summary datasets alongside experiment-level outputs.


***Nov 2024 - Jocelyn Reynolds*** Updated the normalization function to use DI water controls (PD.QN) within each experiment folder to compute normalization factors, enabling more consistent scaling of flow velocity measurements across chips and conditions.

***Dec 2024 - Jocelyn Reynods***, Updated the pipeline to recursively search parent directories and correctly handle multi-part experiment folders (e.g., .P2, .pt2), ensuring that all segments of a dataset are identified and processed together

***Jun 2025- Griffin***, Updated to add Automatic Region of Interest (ROI) determiniation to easlieer identify lane channel location.

    * Resrouces used: OpenCV, Google Gemini LLM

***July 2025- Jocelyn Reynods***, modified code to place CSVs into a seperate directory from the raw data (,mp4 files) and to perfrom single pixel flow profile extraction. Updated Automatic Region of Interest (ROI) to perform multilane batch processing to easily adjsut the number of lanes per channe. Performed 55 lanes per channel to augment data.

***Aug 2025 - Jocelyn Reynolds***, modfified code to change start detection funcion. The updated trimming function uses a variance-based criterion to detect the onset of stable flow, rather than relying on fixed frame thresholds or simple slope estimates. This improves robustness to noise and variability in early-frame signal behavior.

***Sep 2025 - Jocelyn Reynolds***Updated the pipeline to improve computational efficiency by processing all lanes simultaneously per frame, rather than iterating through each lane and frame sequentially

***Dec 2025 - Jocelyn Reynolds***, Fix error with thresholding to determine location of the moving front

***Jan 2026 - Jocelyn Reynolds*** Updated to calculated flow velocity over longer portion of the flow profile (45 seconds) rather than the first 300 frames (10 seconds). Also reverted to the 3 lane averaging with not overlap in averaged lanes


# **Pixel Processing**

In [ ]:
#!/usr/bin/env python3
# coding: utf-8
"""
Bacteria Species Darkening Pipeline
- ROI detection band: 0.35H to 0.65H
- ROI detection method: column bright-count segmentation (pick 4 widest)
- Start-frame detection: absdiff threshold=5 + min_changed_pixels=200 + max_frames=500
- Flow metric: darkening count using FLOW_THRESHOLD=10
- Lanes: one x-column per lane; center-crop to 55 lanes per channel
- Outputs: Raw CSV + Aligned CSV + PDF plots per video
"""

import os
import time
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams["figure.dpi"] = 120

# ========================= CONFIG EDIT THESE =========================
# Set this to the root folder that contains your Bacteria Species videos
BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/Bacteria Species In PBS")

OUTPUT_SUBFOLDER        = "Raw_Flow_Darkening_v2"

USE_CENTER_LANES = True
CENTER_LANES = 55

PRE_SCAN_THRESHOLD = 5
PRE_SCAN_MIN_PIXELS = 200
PRE_SCAN_MAX_FRAMES = 500

FLOW_THRESHOLD = 10

VISUALIZE_ROI_DETECTION = False
VISUALIZE_MOTION_START = False

MAX_WORKERS = 6

FORCE_REPROCESS = False
CLEAN_BEFORE_REPROCESS = False

# ========================= Helper: rotation (optional) =========================
def _rotate_frame(frame, rotate=None):
    if rotate is None:
        return frame
    rotate = str(rotate).lower()
    if rotate in ["90", "cw", "clockwise"]:
        return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    if rotate in ["-90", "ccw", "counterclockwise"]:
        return cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if rotate in ["180"]:
        return cv2.rotate(frame, cv2.ROTATE_180)
    return frame


# ========================= ROI detection (Mixtures-style) =========================
def detect_channel_rois(
    video_path: Path,
    debug_display: bool = False,
    rotate=None,
    bright_thresh: int = 180,
    col_min_bright_pixels: int = 50,
):

    cap = cv2.VideoCapture(str(video_path))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return []

    frame = _rotate_frame(frame, rotate=rotate)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    y_band1 = int(H * 0.35)
    y_band2 = int(H * 0.65)
    band = gray[y_band1:y_band2, :]

    bright = (band > bright_thresh).astype(np.uint8)
    bright_counts = bright.sum(axis=0)

    good_cols = bright_counts > col_min_bright_pixels

    segs = []
    in_seg = False
    start = 0
    for x in range(W):
        if good_cols[x] and not in_seg:
            in_seg = True
            start = x
        elif (not good_cols[x]) and in_seg:
            in_seg = False
            segs.append((start, x))
    if in_seg:
        segs.append((start, W))

    merged = []
    gap_tol = 20
    for s, e in segs:
        if not merged:
            merged.append([s, e])
        else:
            if s - merged[-1][1] <= gap_tol:
                merged[-1][1] = e
            else:
                merged.append([s, e])
    merged = [(a, b) for a, b in merged]

    merged = sorted(merged, key=lambda t: (t[1] - t[0]), reverse=True)[:4]
    merged = sorted(merged, key=lambda t: t[0])

    rois = []
    for (x1, x2) in merged:
        roi_strip = gray[:, x1:x2]
        row_counts = (roi_strip > bright_thresh).sum(axis=1)
        ys = np.where(row_counts > 5)[0]
        if len(ys) < 20:
            continue
        y1 = max(int(ys.min()) - 20, 0)
        y2 = min(int(ys.max()) + 20, H)
        rois.append((x1, x2, y1, y2))

    rois = sorted(rois, key=lambda r: r[2])[:4]

    if debug_display and rois:
        ov = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        for x1, x2, y1, y2 in rois:
            cv2.rectangle(ov, (x1, y1), (x2, y2), (255, 0, 0), 2)
        plt.figure(figsize=(7, 5))
        plt.imshow(ov[..., ::-1])
        plt.title(f"Detected ROIs: {video_path.name}")
        plt.axis("off")
        plt.show()

    return rois


# =========================Start-frame detection =========================
def find_flow_start_frame(
    video_path: Path,
    rois,
    pre_scan_frames: int = PRE_SCAN_MAX_FRAMES,
    threshold: int = PRE_SCAN_THRESHOLD,
    min_changed_pixels: int = PRE_SCAN_MIN_PIXELS,
    rotate=None,
    debug_display: bool = False,
):

    cap = cv2.VideoCapture(str(video_path))
    ok, frame0 = cap.read()
    if not ok:
        cap.release()
        return 0

    frame0 = _rotate_frame(frame0, rotate=rotate)
    base = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY)

    kernel = np.ones((5, 5), np.uint8)

    frame_idx = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frame_idx += 1
        if pre_scan_frames is not None and frame_idx > pre_scan_frames:
            break

        fr = _rotate_frame(fr, rotate=rotate)
        gray = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(gray, base)
        _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        for (x1, x2, y1, y2) in rois:
            roi_mask = mask[y1:y2, x1:x2]
            changed = int(roi_mask.sum() // 255)
            if changed >= min_changed_pixels:
                cap.release()
                if debug_display:
                    print(f"[DEBUG] Start frame detected at {frame_idx} for {video_path.name}")
                return frame_idx

    cap.release()
    return 0


# ========================= Flow extraction =========================
def flowAnalysis_video_full_roi_fast(
    video_path: Path,
    rois,
    threshold: int = FLOW_THRESHOLD,
    use_center_lanes: bool = USE_CENTER_LANES,
    center_lanes: int = CENTER_LANES,
    rotate=None,
    max_frames=None,
):

    cap = cv2.VideoCapture(str(video_path))
    ok, frame0 = cap.read()
    if not ok:
        cap.release()
        return pd.DataFrame()

    frame0 = _rotate_frame(frame0, rotate=rotate)
    gray0 = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY).astype(np.int16)

    lane_specs = []
    lane_names = []

    for ch_idx, (x1, x2, y1, y2) in enumerate(rois, start=1):
        xs = np.arange(x1, x2)

        if use_center_lanes and center_lanes < len(xs):
            c = len(xs) // 2
            half = center_lanes // 2
            xs = xs[c - half : c - half + center_lanes]

        base_roi = gray0[y1:y2, xs]  # (height, n_lanes)

        lane_specs.append(
            {
                "ch": ch_idx,
                "y1": y1,
                "y2": y2,
                "xs": xs,
                "base_roi": base_roi,
                "n": len(xs),
            }
        )

        for lane_i in range(len(xs)):
            lane_names.append(f"Ch{ch_idx}_Lane{lane_i}")

    lane_series = {name: [] for name in lane_names}
    frames = []

    frame_idx = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frame_idx += 1
        if max_frames is not None and frame_idx > max_frames:
            break

        fr = _rotate_frame(fr, rotate=rotate)
        gray = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY).astype(np.int16)
        frames.append(frame_idx)

        lane_offset = 0
        for spec in lane_specs:
            y1, y2, xs = spec["y1"], spec["y2"], spec["xs"]
            base_roi = spec["base_roi"]
            cur_roi = gray[y1:y2, xs]
            counts = np.sum((base_roi - cur_roi) > threshold, axis=0)

            for k, val in enumerate(counts):
                lane_series[lane_names[lane_offset + k]].append(int(val))
            lane_offset += spec["n"]

    cap.release()

    df = pd.DataFrame(lane_series)
    df.insert(0, "Frame", frames)
    return df


def trim_flow_by_start(df: pd.DataFrame, start_frame: int):

    if df.empty or start_frame <= 0:
        return df.copy()

    out = df[df["Frame"] >= start_frame].copy()
    out.reset_index(drop=True, inplace=True)
    out.insert(0, "AlignedFrame", np.arange(len(out)))
    return out


# =========================Plotting=========================
def plot_flow(df, title, pdf=None, frm="Frame"):
    plt.figure(figsize=(8, 4))
    for col in df.columns:
        if col in ["Frame", "AlignedFrame"]:
            continue
        plt.plot(df[frm], df[col], linewidth=0.6)
    plt.gca().set(title=title, xlabel=frm, ylabel="Pixel")
    if pdf:
        pdf.savefig()
        plt.close()
    else:
        plt.show()


def plot_channel_avgs(df, rois, title, pdf=None, frm="Frame"):
    plt.figure(figsize=(8, 4))
    plotted = False
    for ch in range(1, len(rois) + 1):
        ch_cols = [c for c in df.columns if c.startswith(f"Ch{ch}_Lane")]
        if ch_cols:
            plt.plot(df[frm], df[ch_cols].mean(1), label=f"Ch{ch}")
            plotted = True
    plt.gca().set(title=title, xlabel=frm, ylabel="Pixel")
    if plotted:
        plt.legend()
    if pdf:
        pdf.savefig()
        plt.close()
    else:
        plt.show()


def _lock_path_for_video(video_folder: Path, stem: str) -> Path:
    lock_root = video_folder / OUTPUT_SUBFOLDER / ".locks"
    lock_root.mkdir(parents=True, exist_ok=True)
    return lock_root / f"{stem}.lock"


def _try_acquire_lock(lock_path: Path, stale_seconds=6 * 3600) -> bool:
    try:
        if lock_path.exists():
            age = time.time() - lock_path.stat().st_mtime
            if age < stale_seconds:
                return False
            try:
                lock_path.unlink()
            except Exception:
                return False
        lock_path.write_text(f"pid={os.getpid()}\nstart={time.time()}\n")
        return True
    except Exception:
        return False


def _release_lock(lock_path: Path):
    try:
        if lock_path.exists():
            lock_path.unlink()
    except Exception:
        pass


def get_output_paths(video_path: Path):
    base = video_path.parent / OUTPUT_SUBFOLDER
    raw_dir = base / "Raw_CSVs"
    aln_dir = base / "Aligned_CSVs"
    plt_dir = base / "Plots"
    raw_dir.mkdir(parents=True, exist_ok=True)
    aln_dir.mkdir(parents=True, exist_ok=True)
    plt_dir.mkdir(parents=True, exist_ok=True)

    stem = video_path.stem
    raw_csv = raw_dir / f"{stem}_RawFlow.csv"
    aln_csv = aln_dir / f"{stem}_AlignedFlow.csv"
    plot_pdf = plt_dir / f"{stem}_FlowPlots.pdf"
    return raw_csv, aln_csv, plot_pdf


def maybe_clean_outputs(video_path: Path):
    raw_csv, aln_csv, plot_pdf = get_output_paths(video_path)
    for p in [raw_csv, aln_csv, plot_pdf]:
        try:
            if p.exists():
                p.unlink()
        except Exception:
            pass


# ========================= Per-video processing =========================
def process_video(video_path: Path):
    raw_csv, aln_csv, plot_pdf = get_output_paths(video_path)
    stem = video_path.stem

    # Skip unless forcing reprocess
    if (not FORCE_REPROCESS) and aln_csv.exists() and aln_csv.stat().st_size > 200:
        return

    if FORCE_REPROCESS and CLEAN_BEFORE_REPROCESS:
        maybe_clean_outputs(video_path)

    rois = detect_channel_rois(video_path, debug_display=VISUALIZE_ROI_DETECTION)
    if len(rois) < 4:
        print(f" ROI detection failed (<4) for {video_path.name}")
        return

    start = find_flow_start_frame(
        video_path,
        rois,
        pre_scan_frames=PRE_SCAN_MAX_FRAMES,
        threshold=PRE_SCAN_THRESHOLD,
        min_changed_pixels=PRE_SCAN_MIN_PIXELS,
        debug_display=VISUALIZE_MOTION_START,
    )

    df_raw = flowAnalysis_video_full_roi_fast(video_path, rois, threshold=FLOW_THRESHOLD)
    if df_raw.empty:
        print(f" Empty raw df for {video_path.name}")
        return

    df_raw.to_csv(raw_csv, index=False)

    df_aln = trim_flow_by_start(df_raw, start)
    df_aln.to_csv(aln_csv, index=False)

    with PdfPages(plot_pdf) as pdf:
        plot_flow(df_raw, f"Raw (pre-trim), all lanes: {stem}", pdf=pdf, frm="Frame")
        plot_flow(df_aln, f"Aligned (trimmed), all lanes: {stem}", pdf=pdf, frm="AlignedFrame")
        plot_channel_avgs(df_raw, rois, f"Raw (pre-trim) channel averages: {stem}", pdf=pdf, frm="Frame")
        plot_channel_avgs(df_aln, rois, f"Aligned (trimmed) channel averages: {stem}", pdf=pdf, frm="AlignedFrame")

    lane_cols = [c for c in df_raw.columns if c.startswith("Ch")]
    print(f"[INFO] {raw_csv.name}: accepted {len(lane_cols)} lane series for {stem} (start_frame={start})")


def process_video_with_lock(video_path: Path):
    lock_path = _lock_path_for_video(video_path.parent, video_path.stem)
    if not _try_acquire_lock(lock_path):
        return
    try:
        process_video(video_path)
    finally:
        _release_lock(lock_path)


# ========================= Batch processing =========================
def find_videos(folder: Path):
    exts = {".mp4", ".mov", ".avi", ".m4v"}
    vids = []
    for p in folder.rglob("*"):
        if p.suffix.lower() in exts and not p.name.startswith("._"):
            vids.append(p)
    return vids


def main():
    global FORCE_REPROCESS, CLEAN_BEFORE_REPROCESS

    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true", help="Reprocess all videos even if outputs exist")
    parser.add_argument("--clean", action="store_true", help="When used with --force, delete old outputs for each video first")
    parser.add_argument("--workers", type=int, default=MAX_WORKERS, help="Number of parallel workers (default: 6)")
    args = parser.parse_args()

    FORCE_REPROCESS = bool(args.force)
    CLEAN_BEFORE_REPROCESS = bool(args.clean)
    workers = max(1, int(args.workers))

    if not BASE_FOLDER.exists():
        print(f"L BASE_FOLDER not found: {BASE_FOLDER}")
        return

    all_videos = sorted(find_videos(BASE_FOLDER))
    if not all_videos:
        print("No videos found.")
        return

    total_folders = len({v.parent for v in all_videos})
    total_videos = len(all_videos)

    # Determine which need processing
    to_process = []
    for v in all_videos:
        _, aln_csv, _ = get_output_paths(v)
        aln_ok = aln_csv.exists() and aln_csv.stat().st_size > 200
        if FORCE_REPROCESS or (not aln_ok):
            to_process.append(v)

    if len(to_process) == 0:
        print("All videos already processed.")
        return

    print(f"Found {total_videos} videos across {total_folders} folders.")
    print(f"Need processing: {len(to_process)}")
    print(f"Outputs written under: {OUTPUT_SUBFOLDER}")
    print(f"Force reprocess: {FORCE_REPROCESS} | Clean: {CLEAN_BEFORE_REPROCESS} | Workers: {workers}")

    try:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            list(ex.map(process_video_with_lock, to_process))
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] KeyboardInterrupt received. Exiting gracefully.")
        print("Tip: rerun without --force to resume unfinished videos.")
        return

    print("===== Batch Summary =====")
    print(f"Folders scanned:           {total_folders}")
    print(f"Total videos found:        {total_videos}")
    print(f"Videos processed this run: {len(to_process)}")
    print(f"Outputs folder:            {OUTPUT_SUBFOLDER}")
    print("=========================")


if __name__ == "__main__":
    main()


# **Velocity, Trimming, and Normalization**

In [ ]:
#!/usr/bin/env python
# coding: utf-8
'''
# Stage-2 Pipeline

- Export "1 velocity per second" for:
     (A) first 10 seconds
     (B) first 45 seconds
- Keep POS export as-is.
- Keep normalization as-is.

HOW IT WORKS:
- Your Step_Calc uses Lucas-Washburn style: fit y vs sqrt(frame_index)
- To get 1 velocity/sec at 30 fps:
    Step = 30 frames (window length)
    Window edges at 1s,2s,... => k = 30,60,90,...
- We export TWO slope tables per experiment:
     *_aic_10s_1vps.csv  (edges 30..300)
     *_aic_45s_1vps.csv  (edges 30..1350)
'''

from pathlib import Path

# ---- Base folder ----
BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/Bacteria Species In PBS")

# ---- Stage-1 output folder  want to process ----
OUTPUT_SUBFOLDER = "Raw_Flow_Darkening_v2_3laneAverage18Lanes"

ONLY_SUBJECTS = None      # e.g., ["BSA"] or None
ONLY_EXPERIMENTS = None   # e.g., ["03.12.25.AP.PBS..BSA.3"] or None

STAGE2_RESUME = True
STAGE2_FORCE_REPROCESS = False
# =========================Exclusion of certain data folders (old data)=========================

EXCLUDE_GH_FAMILY_FOLDERS = True

EXCLUDE_EXPERIMENT_BASE_KEYS = {
    "10.15.25.JR.PBSRedoCaffeine.3",
}

DROP_OLDER_DUPLICATES_BY_REAGENT_CHIP = False


#========================= Reagent & Bacteria name maps=========================
REAGENT_NAME_MAP = {
    "A": "LL-37",
    "B": "Pep28",
    "G": "CaCl",
    "J": "MgCl",
    "O": "Lysine",
    "U": "BSA",
    "V": "Caffeine",
    "W": "Lysozyme",
    "N": "PBS",
}

REAGENT_FOLDER_TO_CODE = {
    "LL37": "A",
    "LL-37": "A",
    "LL_37": "A",
    "Pep28": "B",
    "CaCl": "G",
    "CaCL": "G",
    "MgCl": "J",
    "MgCL": "J",
    "Lysine": "O",
    "BSA": "U",
    "Caffeine": "V",
    "Lysozyme": "W",
    "PBS": "N",
}

BACTERIA_NAME_MAP = {
    "R": "S. aureus",
    "K": "CoNS",
    "D": "PBS / DI water",
}

def _label_for_reagent(rcode_or_name):
    s = str(rcode_or_name).strip()
    if s == "":
        return "Unknown"
    nm = REAGENT_NAME_MAP.get(s.upper(), s)
    return str(nm).replace(" ", "_")

# ========================="1 velocity per second" controls=========================
FPS = 30
EXPORT_1VPS_10S = True
EXPORT_1VPS_45S = True

# =========================Imports=========================

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["figure.dpi"] = 120


def _parse_date_from_exp_name(exp_name: str):
    import datetime as _dt
    s = str(exp_name)
    m = re.match(r"^\s*(\d{1,2})\.(\d{1,2})\.(\d{2})\.", s)
    if not m:
        return None
    mm, dd, yy = int(m.group(1)), int(m.group(2)), int(m.group(3))
    year = 2000 + yy if yy < 80 else 1900 + yy
    try:
        return _dt.date(year, mm, dd)
    except Exception:
        return None

def _parse_part_number(exp_name: str) -> int:
    m = re.search(r'\.(?:p|pt)(\d+)$', str(exp_name), flags=re.IGNORECASE)
    return int(m.group(1)) if m else 1

def _strip_part_suffix(exp_name: str) -> str:
    return re.sub(r'\.(?:p|pt)\d+$', '', str(exp_name), flags=re.IGNORECASE)

def _is_gh_family_folder(exp_name: str) -> bool:
    toks = [t.upper() for t in str(exp_name).split(".") if t]
    return any(t in ("GH", "GHD", "GDH") for t in toks)

def _parse_reagent_name_from_exp_name(exp_name: str):
    """
    reagent token is the dot-token immediately BEFORE the chip number,
    where chip number is the last standalone integer before optional .P2/.pt2.
    (Used only as a fallback; truth is reagent folder name.)
    """
    toks = [t for t in str(exp_name).split(".") if t != ""]
    if len(toks) < 2:
        return None

    if re.match(r"^(?:P|pt)\d+$", toks[-1], flags=re.IGNORECASE):
        toks = toks[:-1]

    chip_idx = None
    for i in range(len(toks)-1, -1, -1):
        if toks[i].isdigit():
            chip_idx = i
            break
    if chip_idx is None or chip_idx == 0:
        return None
    return toks[chip_idx - 1]

def _parse_chip_from_exp_name(exp_name: str):
    toks = [t for t in str(_strip_part_suffix(exp_name)).split(".") if t != ""]
    for i in range(len(toks)-1, -1, -1):
        if toks[i].isdigit():
            return int(toks[i])
    return None

def _apply_exclusions_and_dedup(folders):
    kept = []
    dropped_gh = 0
    dropped_base = 0

    for (subject, experiment, data_dir, raw_csv_dir) in folders:
        exp_name = experiment.name
        if EXCLUDE_GH_FAMILY_FOLDERS and _is_gh_family_folder(exp_name):
            dropped_gh += 1
            continue
        base_key = _strip_part_suffix(exp_name)
        if base_key in set(EXCLUDE_EXPERIMENT_BASE_KEYS):
            dropped_base += 1
            continue
        kept.append((subject, experiment, data_dir, raw_csv_dir))

    if dropped_gh:
        print(f"🚫 Excluded {dropped_gh} folder(s) due to GH/GHD/GDH token rule.")
    if dropped_base:
        print(f"🚫 Excluded {dropped_base} folder(s) due to explicit base-key exclusions.")

    if not DROP_OLDER_DUPLICATES_BY_REAGENT_CHIP:
        return kept

    best = {}
    dups_dropped = 0

    for item in kept:
        subject, experiment, data_dir, raw_csv_dir = item
        exp_name = experiment.name
        reagent = _parse_reagent_name_from_exp_name(exp_name) or "UnknownReagent"
        chip = _parse_chip_from_exp_name(exp_name)
        dt = _parse_date_from_exp_name(exp_name)
        key = (str(reagent), int(chip) if chip is not None else None)

        if key not in best:
            best[key] = (dt, item)
            continue

        old_dt, _old_item = best[key]

        if old_dt is None and dt is not None:
            best[key] = (dt, item); dups_dropped += 1; continue
        if old_dt is not None and dt is None:
            dups_dropped += 1; continue

        if (old_dt is not None) and (dt is not None):
            if dt > old_dt:
                best[key] = (dt, item); dups_dropped += 1
            else:
                dups_dropped += 1
            continue

        dups_dropped += 1

    out = [v[1] for v in best.values()]
    if dups_dropped:
        print(f"🧹 Dropped {dups_dropped} folder(s) as older duplicates per (Reagent, Chip) newest-date rule.")
    return sorted(out, key=lambda t: (t[0].name, t[1].name))


def _iter_raw_csv_folders_humans(base_dir):

    base_dir = Path(base_dir)
    for subject in sorted(p for p in base_dir.iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ONLY_SUBJECTS and subject.name not in set(ONLY_SUBJECTS):
            continue
        for experiment in sorted(p for p in subject.iterdir() if p.is_dir() and not p.name.startswith(".")):
            if ONLY_EXPERIMENTS and experiment.name not in set(ONLY_EXPERIMENTS):
                continue
            data_dir = experiment / "Data"
            raw_csv_dir = data_dir / OUTPUT_SUBFOLDER / "Raw_CSVs"
            if raw_csv_dir.exists() and raw_csv_dir.is_dir():
                yield (subject, experiment, data_dir, raw_csv_dir)

def _safe_first(df, col, default=None):
    return df[col].iloc[0] if (col in df.columns and len(df) > 0) else default

def _base_stem_from_filename(stem: str) -> str:
    s = str(stem)
    s = s.replace("_RawFlow", "").replace("_AlignedFlow", "")
    return s

def _parse_chip_from_stem(stem: str):
    nums = re.findall(r'\.(\d+)', str(stem))
    return int(nums[-1]) if nums else None

def _video_stem_to_combo(stem):
    s = str(stem)
    m = re.search(r"P([A-Za-z])\.Q([A-Za-z])\.(\d+)", s, flags=re.IGNORECASE)
    if m:
        return "P%s.Q%s.%s" % (m.group(1).upper(), m.group(2).upper(), m.group(3))
    m3 = re.search(r"^(P[A-Za-z])\.(Q[A-Za-z])\.(\d+)$", s, flags=re.IGNORECASE)
    if m3:
        return "%s.%s.%s" % (m3.group(1).upper(), m3.group(2).upper(), m3.group(3))
    parts = re.split(r"[._\\-]+", s)
    bac, rea, chip = None, None, None
    for tok in parts:
        if tok.startswith("P") and len(tok) >= 2 and tok[1].isalpha():
            bac = tok[:2].upper()
        if tok.startswith("Q") and len(tok) >= 2 and tok[1].isalpha():
            rea = tok[:2].upper()
        if tok.isdigit():
            chip = tok
    return "%s.%s.%s" % (bac, rea, chip) if (bac and rea and chip) else s

def _ensure_pdf(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return PdfPages(path)

def _experiment_output_paths(easyName, experiment_folder):
    pos = Path(experiment_folder) / f"{easyName}_Raw_Average_Pos.csv"
    aic_10s = Path(experiment_folder) / f"{easyName}_Raw_Average_aic_10s_1vps.csv"
    aic_45s = Path(experiment_folder) / f"{easyName}_Raw_Average_aic_45s_1vps.csv"
    aic25 = Path(experiment_folder) / f"{easyName}_Raw_Average_aic25s.csv"
    return pos, aic_10s, aic_45s, aic25

def _experiment_already_done(easyName, experiment_folder):
    pos, a10, a45, aic25 = _experiment_output_paths(easyName, experiment_folder)
    need = [pos]
    if EXPORT_1VPS_10S: need.append(a10)
    if EXPORT_1VPS_45S: need.append(a45)
    if EXPORT_LEGACY_AIC25S: need.append(aic25)
    return all(p.exists() for p in need)

def _load_existing_norm_tables(easyName, experiment_folder):
    _, a10, a45, aic25 = _experiment_output_paths(easyName, experiment_folder)

    def _load(path):
        path = Path(path)
        if not path.exists():
            return pd.DataFrame()
        df = pd.read_csv(path)

        if "ReagentCode" not in df.columns or df["ReagentCode"].isna().all():
            if "Combo" in df.columns:
                rc = df["Combo"].astype(str).str.extract(r"Q([A-Za-z])", expand=False).str.upper()
                df["ReagentCode"] = rc

        if "ReagentName" not in df.columns:
            df["ReagentName"] = df.get("ReagentCode", pd.Series(dtype=object)).map(
                lambda r: REAGENT_NAME_MAP.get(str(r).upper(), r)
            )

        if "BacteriaName" not in df.columns:
            bac = df.get("Combo", pd.Series(dtype=object)).astype(str).str.extract(r"P([A-Za-z])", expand=False).str.upper()
            df["BacteriaName"] = bac.map(lambda b: BACTERIA_NAME_MAP.get(str(b), b))

        return df

    out = {}
    if EXPORT_1VPS_10S: out["a10"] = _load(a10)
    if EXPORT_1VPS_45S: out["a45"] = _load(a45)
    if EXPORT_LEGACY_AIC25S: out["aic25"] = _load(aic25)
    return out

def _resolve_reagent_from_context(reagent_folder_name: str, easyName: str, df: pd.DataFrame):

    folder = str(reagent_folder_name).strip()
    folder_key = folder.replace(" ", "").replace("_", "").replace("-", "")
    folder_code = REAGENT_FOLDER_TO_CODE.get(folder) or REAGENT_FOLDER_TO_CODE.get(folder_key) or None
    folder_name = REAGENT_NAME_MAP.get(folder_code, folder) if folder_code else folder

    csv_code = str(_safe_first(df, "ReagentCode") or "").strip().upper() or None
    csv_name = _safe_first(df, "ReagentName")
    csv_name = str(csv_name).strip() if csv_name is not None else None
    csv_name = csv_name if csv_name not in ("", "nan", "None") else None

    exp_tok = _parse_reagent_name_from_exp_name(easyName)
    exp_tok = str(exp_tok).strip() if exp_tok else None
    exp_code = None
    if exp_tok:
        exp_key = exp_tok.replace(" ", "").replace("_", "").replace("-", "")
        exp_code = REAGENT_FOLDER_TO_CODE.get(exp_tok) or REAGENT_FOLDER_TO_CODE.get(exp_key) or None
    exp_name = REAGENT_NAME_MAP.get(exp_code, exp_tok) if exp_tok else None

    reagent_code = folder_code or csv_code or exp_code
    reagent_name = folder_name or csv_name or exp_name or "Reagent"
    reagent_name = str(reagent_name).strip()
    if reagent_name in ("", "nan", "None"):
        reagent_name = "Reagent"
    return reagent_code, reagent_name

# ========================= Main pipeline =========================


def ProcessRawFlow(base_dir, show_inline=True):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError("[Stage-2] Base directory not found: %s" % base_dir)

    global_pdf_path = base_dir / "Stage2_Global" / "Consolidated_Raw_Average_TrimVerification.pdf"
    global_pdf = _ensure_pdf(global_pdf_path)

    per_reagent = {"a10": {}, "a45": {}, "aic25": {}}
    (base_dir / "Stage2_Scratch").mkdir(parents=True, exist_ok=True)

    folders = list(_iter_raw_csv_folders_humans(base_dir))
    print("[PATH CHECK] BASE_FOLDER exists? %s" % base_dir.exists())
    print("Found %d experiment folder(s) with %s CSVs under: %s"
          % (len(folders), OUTPUT_SUBFOLDER, base_dir))

    folders = _apply_exclusions_and_dedup(folders)
    total_folders = len(folders)
    print("After exclusions/de-dup: %d experiment folder(s) remain." % total_folders)

    folders_found = 0
    for idx, (subject_path, experiment_path, data_path, raw_csv_dir) in enumerate(folders, start=1):
        folders_found += 1

        reagent_folder_name = subject_path.name

        subject = subject_path.name
        easyName = experiment_path.name
        experiment_folder = experiment_path

        print("\n Processing folder %d/%d: %s / %s" % (idx, total_folders, subject, easyName))

        part_num = _parse_part_number(experiment_path.name)


        if STAGE2_RESUME and (not STAGE2_FORCE_REPROCESS) and _experiment_already_done(easyName, experiment_folder):
            print(" Skipping compute (valid exports already exist).")
            loaded = _load_existing_norm_tables(easyName, experiment_folder)

            for key, df in loaded.items():
                if df is None or df.empty or ("ReagentCode" not in df.columns):
                    continue
                for rcode, block in df.groupby("ReagentCode"):
                    per_reagent[key].setdefault(str(rcode), []).append(block.copy())

            continue

        open_reagent_pdfs = {}
        all_data = {"title": easyName}
        seen_preview_titles = set()

        resolved_reagent_code = None
        resolved_reagent_name = None

        csv_files = sorted(p for p in raw_csv_dir.glob("*_RawFlow.csv") if p.is_file())
        if not csv_files:
            print("   (no *_RawFlow.csv files found here)")
            continue

        for csvf in csv_files:
            base = csvf.stem
            if base.startswith("fixed_"):
                continue

            try:
                df = pd.read_csv(csvf)

                reagent_code, reagent_name = _resolve_reagent_from_context(
                    reagent_folder_name=reagent_folder_name,
                    easyName=easyName,
                    df=df
                )
                resolved_reagent_code = reagent_code
                resolved_reagent_name = reagent_name
                reagent_label = resolved_reagent_name

                base_stem = _base_stem_from_filename(csvf.stem)
                chip_num  = _parse_chip_from_stem(base_stem)

                video_stem = _safe_first(df, "VideoStem") or base_stem
                combo_key = _video_stem_to_combo(video_stem)

                parsed_title = f"{combo_key}.Chip{chip_num if chip_num is not None else 'X'}.Part{part_num}"

                parsed = define_df_from_flow_csv(df, title=parsed_title, log_prefix=csvf.name)

                trimmed, trim_lines, raw_trim_map = trim_by_channel_average(parsed)

                all_data[trimmed["title"]] = trimmed

                pdf_key = (resolved_reagent_code or resolved_reagent_name or "Reagent")
                if pdf_key and (pdf_key not in open_reagent_pdfs):
                    file_label = _label_for_reagent(pdf_key) if resolved_reagent_code else str(resolved_reagent_name).replace(" ", "_")
                    rpdf_path = base_dir / "Stage2_PerReagent" / (f"{file_label}_Raw_Average_TrimVerification.pdf")
                    open_reagent_pdfs[pdf_key] = _ensure_pdf(rpdf_path)

                if parsed_title not in seen_preview_titles:
                    show_both_panels(
                        parsed, trimmed, parsed_title,
                        reagent_label,
                        pdf_local=open_reagent_pdfs.get(pdf_key),
                        pdf_global=global_pdf,
                        show_inline=show_inline,
                        trim_lines=trim_lines
                    )
                    seen_preview_titles.add(parsed_title)

            except Exception as e:
                print("[ERROR] Failed %s: %s" % (csvf.name, e))

        f_list  = find_factor_list(all_data)
        combined = combine_chip_data(all_data, f_list)

        export_csv(pd.DataFrame(combined), f"{easyName}_Raw_Average_Pos.csv", experiment_folder)

        reagent_code_override = resolved_reagent_code or REAGENT_FOLDER_TO_CODE.get(reagent_folder_name) or None
        reagent_name_override = resolved_reagent_name or reagent_folder_name
        reagent_token_override = ("Q" + str(reagent_code_override)) if reagent_code_override else None


        if EXPORT_1VPS_10S:
            slopes_10s = Step_Calc(
                combined,
                Step=FPS,
                numSteps=10,  # edges 30..300
                reagent_code_override=reagent_code_override,
                reagent_name_override=reagent_name_override,
                reagent_token_override=reagent_token_override,
            )
            norm_10s = Normalize_Slope(slopes_10s)
            if not norm_10s.empty:
                export_csv(norm_10s, f"{easyName}_Raw_Average_aic_10s_1vps.csv", experiment_folder)
                if "ReagentCode" in norm_10s.columns:
                    for rcode, block in norm_10s.groupby("ReagentCode"):
                        per_reagent["a10"].setdefault(str(rcode), []).append(block.copy())

        if EXPORT_1VPS_45S:
            slopes_45s = Step_Calc(
                combined,
                Step=FPS,
                numSteps=45,  # edges 30..1350
                reagent_code_override=reagent_code_override,
                reagent_name_override=reagent_name_override,
                reagent_token_override=reagent_token_override,
            )
            norm_45s = Normalize_Slope(slopes_45s)
            if not norm_45s.empty:
                export_csv(norm_45s, f"{easyName}_Raw_Average_aic_45s_1vps.csv", experiment_folder)
                if "ReagentCode" in norm_45s.columns:
                    for rcode, block in norm_45s.groupby("ReagentCode"):
                        per_reagent["a45"].setdefault(str(rcode), []).append(block.copy())

        if EXPORT_LEGACY_AIC25S:
            slopes_df_25 = Step_Calc(
                combined,
                numSteps=50,
                reagent_code_override=reagent_code_override,
                reagent_name_override=reagent_name_override,
                reagent_token_override=reagent_token_override,
            )
            norm25 = Normalize_Slope(slopes_df_25)
            if not norm25.empty:
                export_csv(norm25, f"{easyName}_Raw_Average_aic25s.csv", experiment_folder)
                if "ReagentCode" in norm25.columns:
                    for rcode, block in norm25.groupby("ReagentCode"):
                        per_reagent["aic25"].setdefault(str(rcode), []).append(block.copy())

        for _, rpdf in open_reagent_pdfs.items():
            try:
                rpdf.close()
            except Exception:
                pass

    per_reagent_root = base_dir / "Stage2_PerReagent"
    per_reagent_root.mkdir(parents=True, exist_ok=True)

    def _write_rollups(key, suffix):
        for rcode, blocks in per_reagent[key].items():
            out_csv = per_reagent_root / (f"{_label_for_reagent(rcode)}_All_Raw_Average_{suffix}.csv")
            try:
                pd.concat(blocks, ignore_index=True).to_csv(out_csv, index=False)
                print("🧾 Wrote: %s" % out_csv.name)
            except Exception as e:
                print("[WARN] Could not write %s: %s" % (out_csv.name, e))

    if EXPORT_1VPS_10S:
        _write_rollups("a10", "aic_10s_1vps")
    if EXPORT_1VPS_45S:
        _write_rollups("a45", "aic_45s_1vps")
    if EXPORT_LEGACY_AIC25S:
        _write_rollups("aic25", "aic25s")

    try:
        global_pdf.close()
    except Exception:
        pass

    print(" Finished Stage-2 processing. Folders scanned: %d of %d" % (folders_found, total_folders))


def define_df_from_flow_csv(df, title, log_prefix=""):
    chip_data = {'title': title}
    accepted = []
    for col in df.columns:
        if col in ("Frame", "AlignedFrame"):
            continue
        if not (str(col).startswith("C") or str(col).startswith("Ch")):
            continue
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        if series.empty:
            continue
        chip_data[col] = series.tolist()
        accepted.append(col)

    if log_prefix:
        print("[INFO] %s: accepted %d lane series for %s" % (log_prefix, len(accepted), title))
    else:
        print("[INFO] accepted %d lane series for %s" % (len(accepted), title))
    return chip_data

def find_factor_list(large_data_dict):
    combos = set()
    for k in large_data_dict:
        if k == "title":
            continue
        base = k.split(".Chip", 1)[0]
        combos.add(base)
    return sorted(combos)

def combine_chip_data(large_data, f_list):
    combined = {"title": large_data.get("title", "Untitled")}
    for combo in f_list:
        curr = {}
        for key in large_data:
            if key == "title":
                continue
            if str(key).startswith(combo + ".Chip"):
                for lane_key in (k for k in large_data[key] if k != "title"):
                    curr[lane_key] = large_data[key][lane_key]
        if curr:
            combined[combo] = curr
    return combined

def Step_Calc(
    data_input,
    Step=15,
    return_table=False,
    output_edge=0,
    numSteps=20,
    reagent_code_override=None,
    reagent_name_override=None,
    reagent_token_override=None,
):
    """
    Lucas-Washburn style: fit y vs sqrt(frame_index) in windows.
    """
    FitScale = np.arange(Step, (numSteps + 1) * Step, step=Step)

    cols = [
        "Slope","WindowEdge","Combo","Plastic","Reagent","Channel","Experiment Folder",
        "ReagentCode","ReagentName","BacteriaName"
    ]
    out = pd.DataFrame(columns=cols)
    exp_title = data_input.get("title", "UnknownFolder")

    rcode_fix = (str(reagent_code_override).strip().upper() if reagent_code_override else None)
    rname_fix = (str(reagent_name_override).strip() if reagent_name_override else None)
    if rname_fix in ("", "nan", "None"):
        rname_fix = None

    for combo_key in [k for k in data_input if k != "title"]:
        m = re.search(r"(P[A-Z])\.?(Q[A-Z]+)", str(combo_key))
        if m:
            combo_lbl = "%s.%s" % (m.group(1), m.group(2))
            plastic = m.group(1)[1:]
            reagent_token_from_combo = m.group(2)  # e.g., QH (concentration token)
            bacteria_name = BACTERIA_NAME_MAP.get(plastic, plastic)
        else:
            combo_lbl, plastic, reagent_token_from_combo = "PX.QX", "X", "QX"
            bacteria_name = None

        reagent_code = rcode_fix
        reagent_name = (REAGENT_NAME_MAP.get(reagent_code, reagent_code) if reagent_code else None)
        if rname_fix:
            reagent_name = rname_fix

        if reagent_token_override:
            reagent = str(reagent_token_override)
        elif reagent_code:
            reagent = "Q" + str(reagent_code)
        else:
            reagent = reagent_token_from_combo

        for chan in data_input[combo_key]:
            if chan == "title":
                continue
            data = np.asarray(data_input[combo_key][chan], dtype=float)
            x = np.arange(1, len(data) + 1)

            for k in FitScale:
                if k > len(data):
                    continue
                xx = np.sqrt(x[k-Step:k])
                yy = data[k-Step:k]
                slope = np.polyfit(xx, yy, 1)[0] if len(xx) >= 2 else np.nan

                out.loc[len(out)] = [
                    slope, k, combo_lbl, plastic, reagent, chan, exp_title,
                    reagent_code, reagent_name, bacteria_name
                ]

    return out[out["WindowEdge"] == output_edge] if return_table else out

def Normalize_Slope(df, agg="mean", require_positive_di=False, min_reps=1):
    if df is None or df.empty:
        print("[WARN] No slope data to normalize.")
        return pd.DataFrame()

    out_chunks = []
    for folder, g in df.groupby("Experiment Folder", dropna=False):
        g = g.copy()
        g["Slope"] = pd.to_numeric(g["Slope"], errors="coerce")

        combo_is_di  = g["Combo"].astype(str).str.contains(r"PD\.?\s*QN", case=False, regex=True, na=False)
        plastic_is_d = g["Plastic"].astype(str).str.upper().eq("D")
        di = g.loc[combo_is_di | plastic_is_d].copy()

        if di.empty:
            print("[WARN] No DI water found in %s. Skipping normalization." % folder)
            out_chunks.append(g)
            continue

        g["Normalization Factor"] = np.nan
        g["Normalized_Slope"] = np.nan

        for edge in np.sort(g["WindowEdge"].dropna().unique()):
            di_edge = pd.to_numeric(di.loc[di["WindowEdge"] == edge, "Slope"], errors="coerce")
            di_edge = di_edge[np.isfinite(di_edge)]
            if require_positive_di:
                di_edge = di_edge[di_edge > 0]
            if len(di_edge) < min_reps:
                print("[WARN] %s – Not enough DI for edge %s (n=%d)." % (folder, int(edge), len(di_edge)))
                continue

            di_val = np.median(di_edge) if str(agg).lower() == "median" else float(np.mean(di_edge))
            if not np.isfinite(di_val):
                continue

            mask_g = (g["WindowEdge"] == edge)
            g.loc[mask_g, "Normalization Factor"] = di_val
            g.loc[mask_g, "Normalized_Slope"] = g.loc[mask_g, "Slope"] / di_val
            print("[INFO] %s – Edge %s normalized by DI %s %.4f" % (folder, int(edge), agg, di_val))

        out_chunks.append(g)

    return pd.concat(out_chunks, ignore_index=True)

def export_csv(df, filename, folder):
    folder = Path(folder)
    if df is None or df.empty:
        print("[WARN] Skipping empty export: %s" % filename)
        return
    folder.mkdir(parents=True, exist_ok=True)
    p = folder / filename
    if p.exists():
        p.unlink()
    df.to_csv(p, index=False)
    print("[EXPORT] %s" % p)


def trim_by_channel_average(original_dict,
                            valley_min_start=140,
                            valley_search_end=380,
                            smooth_window=31,
                            slope_win=15,
                            slope_thr=0.35,
                            confirm_len=25,
                            rise_min=20,
                            search_forward=180,
                            consensus_mode="clamped",
                            combine_method="p90",
                            outlier_sigma=2.5,
                            max_channel_offset=20,
                            start_offset=10,
                            frame_end=1400):
    cleaned = {"title": original_dict.get("title", "Untitled")}
    groups = _group_lanes_by_channel(original_dict)
    means = _channel_mean_series(groups)

    raw_trim_map = {}
    for ch_label, avg in means.items():
        s = detect_start_hybrid(
            avg,
            valley_min_start=valley_min_start,
            valley_search_end=valley_search_end,
            smooth_window=smooth_window,
            slope_win=slope_win,
            slope_thr=slope_thr,
            confirm_len=confirm_len,
            rise_min=rise_min,
            search_forward=search_forward,
        )
        raw_trim_map[ch_label] = int(s)

    consensus = None
    if consensus_mode in ("global", "clamped"):
        consensus = robust_consensus_start(list(raw_trim_map.values()), method=combine_method, outlier_sigma=outlier_sigma)
        consensus = int(consensus) + int(start_offset)

    applied_starts = {}
    if consensus_mode == "global" and consensus is not None:
        for ch_label in means.keys():
            applied_starts[ch_label] = consensus
    else:
        for ch_label, s in raw_trim_map.items():
            s_eff = int(s) + int(start_offset)
            if consensus_mode == "clamped" and consensus is not None:
                lo = consensus - int(max_channel_offset)
                hi = consensus + int(max_channel_offset)
                s_eff = int(np.clip(s_eff, lo, hi))
            applied_starts[ch_label] = s_eff

    per_lane_trimmed = {}
    for ch_num, lanes in groups.items():
        ch_label = "Ch%d" % ch_num
        s_idx = int(applied_starts.get(ch_label, 0))
        lane_keys = _lane_keys_for_channel(original_dict, ch_num)
        for lane_pos, arr in enumerate(lanes, start=1):
            arr = np.asarray(arr, dtype=float)
            trimmed = arr[s_idx:] if s_idx < len(arr) else np.array([], dtype=float)
            if frame_end is not None and len(trimmed) > int(frame_end):
                trimmed = trimmed[:int(frame_end)]
            key = lane_keys[lane_pos-1] if (lane_pos-1) < len(lane_keys) else "C%d_P%d" % (ch_num, lane_pos)
            per_lane_trimmed[key] = trimmed

    valid = [v for v in per_lane_trimmed.values() if isinstance(v, np.ndarray) and v.size > 0]
    if valid:
        L = min(len(v) for v in valid)
        for k, v in per_lane_trimmed.items():
            if isinstance(v, np.ndarray) and v.size >= L:
                cleaned[k] = v[:L].tolist()

    if consensus_mode == "global" and consensus is not None:
        trim_lines = {"Global": int(consensus)}
    else:
        trim_lines = {k: int(v) for k, v in applied_starts.items()}

    return cleaned, trim_lines, raw_trim_map

def detect_start_hybrid(avg,
                        valley_min_start=80,
                        valley_search_end=300,
                        smooth_window=31,
                        slope_win=15,
                        slope_thr=0.35,
                        confirm_len=25,
                        rise_min=20,
                        search_forward=180):
    a = np.asarray(avg, dtype=float)
    if a.size == 0:
        return 0

    a_sm = rolling_mean(a, int(smooth_window))
    lo = max(0, int(valley_min_start))
    hi = min(len(a_sm), int(valley_search_end))
    valley = int(lo + np.argmin(a_sm[lo:hi])) if (hi - lo) >= 5 else int(lo)

    d = np.diff(a_sm, prepend=a_sm[0])
    d_sm = rolling_mean(d, int(slope_win))

    def passes_upturn(idx):
        j1, j2 = int(idx), min(len(d_sm), int(idx) + int(confirm_len))
        if (j2 - j1) < int(confirm_len):
            return False
        strong_slope = np.all(d_sm[j1:j2] > float(slope_thr))
        k2 = min(len(a_sm), int(idx) + int(search_forward))
        enough_rise = (a_sm[k2 - 1] - a_sm[int(idx)]) >= float(rise_min) if k2 > int(idx) else False
        return bool(strong_slope and enough_rise)

    if passes_upturn(valley):
        return int(valley)

    for idx in range(int(valley) + 1, min(len(a_sm) - int(confirm_len), int(valley) + int(search_forward))):
        if passes_upturn(idx):
            return int(idx)

    return int(valley)

def robust_consensus_start(starts, method="p95", outlier_sigma=2.5):
    vals = np.asarray([int(s) for s in starts if np.isfinite(s)], dtype=float)
    if vals.size == 0:
        return 0
    med = np.median(vals)
    mad = np.median(np.abs(vals - med))
    scale = max(1e-6, 1.4826 * mad)
    keep = np.abs(vals - med) <= float(outlier_sigma) * scale
    filt = vals[keep] if keep.any() else vals

    if method == "median":
        c = np.median(filt)
    elif method == "p90":
        c = np.percentile(filt, 90)
    elif method == "max":
        c = np.max(filt)
    else:
        c = np.percentile(filt, 95)
    return int(round(float(c)))

def _group_lanes_by_channel(data_dict):
    groups = {}
    for k, arr in data_dict.items():
        if k == "title":
            continue
        if not isinstance(arr, (list, tuple, np.ndarray)):
            continue
        m = re.match(r"^(?:C|Ch)(\d+)[._-]?", str(k), flags=re.IGNORECASE)
        if not m:
            continue
        ch = int(m.group(1))
        groups.setdefault(ch, []).append(np.asarray(arr, dtype=float))
    return groups

def _lane_keys_for_channel(original_dict, ch_num):
    keys = [k for k in original_dict.keys()
            if k != "title" and re.match(r"^(?:C|Ch)%d[._-]?P?\d+" % int(ch_num), str(k), flags=re.IGNORECASE)]
    def lane_index(k):
        m = re.search(r"P(\d+)", str(k), flags=re.IGNORECASE)
        return int(m.group(1)) if m else 9999
    return sorted(keys, key=lane_index)

def _channel_mean_series(groups):
    means = {}
    for ch, lanes in sorted(groups.items()):
        lanes = [x for x in lanes if x.size > 0]
        if not lanes:
            continue
        L = min(len(x) for x in lanes)
        stack = np.vstack([x[:L] for x in lanes])
        means["Ch%d" % ch] = stack.mean(axis=0)
    return means

def rolling_mean(x, w):
    x = np.asarray(x, dtype=float)
    if w <= 1 or x.size == 0:
        return x.astype(float)
    k = np.ones(int(w), dtype=float) / float(w)
    return np.convolve(x.astype(float), k, mode="same")

def show_both_panels(parsed, trimmed, parsed_title, reagent_label,
                     pdf_local=None, pdf_global=None, show_inline=True, trim_lines=None):
    import re
    import numpy as np
    import matplotlib.pyplot as plt
    from collections import defaultdict

    def _chan_num(key):
        m = re.match(r"^(?:C|Ch)(\d+)", str(key), flags=re.IGNORECASE)
        return int(m.group(1)) if m else None

    def _lane_keys(dct):
        return [k for k in dct.keys() if k != "title"]

    base_colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", []) or \
                  ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]

    lanes_raw  = _lane_keys(parsed)
    lanes_trim = _lane_keys(trimmed)
    lane_keys_all = sorted(set(lanes_raw) | set(lanes_trim), key=lambda s: (_chan_num(s) or 9999, s))
    lane_color = {lk: base_colors[i % len(base_colors)] for i, lk in enumerate(lane_keys_all)}

    def _color_for_channel(ch):
        return base_colors[(int(ch) - 1) % len(base_colors)]

    fig = plt.figure(figsize=(12, 9), constrained_layout=False)
    fig.subplots_adjust(left=0.07, right=0.98, bottom=0.08, top=0.84)

    gs = fig.add_gridspec(2, 2, hspace=0.28, wspace=0.18)
    ax11 = fig.add_subplot(gs[0, 0])
    ax12 = fig.add_subplot(gs[0, 1])
    ax21 = fig.add_subplot(gs[1, 0])
    ax22 = fig.add_subplot(gs[1, 1])

    def _plot_all_lanes(ax, dct, title, show_guides):
        for lk in lane_keys_all:
            if lk not in dct:
                continue
            y = np.asarray(dct[lk], dtype=float)
            if y.size == 0:
                continue
            ax.plot(y, lw=0.8, alpha=0.9, color=lane_color[lk])
        if show_guides and trim_lines and isinstance(trim_lines, dict):
            for _, x in trim_lines.items():
                ax.axvline(int(x), ls="--", color="crimson", alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel("Frame")
        ax.set_ylabel("Pixel")

    def _channel_means(dct):
        stacks = defaultdict(list)
        for lk in dct:
            if lk == "title":
                continue
            ch = _chan_num(lk)
            if ch is None:
                continue
            arr = np.asarray(dct[lk], dtype=float)
            if arr.size == 0:
                continue
            stacks[ch].append(arr)
        means = {}
        for ch, arrs in stacks.items():
            L = min(len(a) for a in arrs)
            if L <= 0:
                continue
            stack = np.vstack([a[:L] for a in arrs])
            means[ch] = stack.mean(axis=0)
        return dict(sorted(means.items(), key=lambda kv: kv[0]))

    def _plot_means(ax, dct, title, show_guides):
        means = _channel_means(dct)
        for ch, y in means.items():
            ax.plot(y, lw=2.2, color=_color_for_channel(ch), label="Ch%d" % ch)
        if show_guides and trim_lines and isinstance(trim_lines, dict):
            for _, x in trim_lines.items():
                ax.axvline(int(x), ls="--", color="crimson", alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel("Frame")
        ax.set_ylabel("Pixel")
        return list(means.keys())

    _plot_all_lanes(ax11, parsed,  "Raw (pre-trim), all lanes: %s" % parsed_title, show_guides=True)
    _plot_all_lanes(ax12, trimmed, "Aligned (trimmed), all lanes: %s" % parsed_title, show_guides=False)

    chs_left  = _plot_means(ax21, parsed,  "Raw (pre-trim) channel averages", show_guides=True)
    chs_right = _plot_means(ax22, trimmed, "Aligned (trimmed) channel averages", show_guides=False)

    channel_set = sorted(set(chs_left) | set(chs_right))
    handles = []
    for ch in channel_set:
        h, = ax22.plot([], [], label="Ch%d" % ch, color=_color_for_channel(ch), lw=2.2)
        handles.append(h)
    if handles:
        fig.legend(handles=handles, loc="upper center", ncol=min(6, len(handles)),
                   frameon=False, bbox_to_anchor=(0.5, 0.985))

    fig.text(0.5, 0.94, "Reagent: %s" % reagent_label, ha="center", va="center", fontsize=13)

    if pdf_local:
        pdf_local.savefig(fig)
    if pdf_global:
        pdf_global.savefig(fig)
    if show_inline:
        plt.show()
    plt.close(fig)

if __name__ == "__main__":
    print("[PATH CHECK] BASE_FOLDER exists? %s" % BASE_FOLDER.exists())
    ProcessRawFlow(BASE_FOLDER, show_inline=True)


# **Building ML Table**

In [ ]:
'''#!/usr/bin/env python
# coding: utf-8
 ML TABLE BUILDER (Bacteria Species In PBS)
 - Loads exported slope CSVs from experiment folders
 - Uses reagent folder name as truth for reagent identity
 - Applies CONDITION-LEVEL redo override: condition=(Combo,ChipNumber)
 - Builds aligned pivot table across reagents (inner join)
'''

from pathlib import Path
import re
import pandas as pd
from functools import reduce

# ========================= CONFIG =========================
BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/Bacteria Species In PBS")

OUTPUT_SUBFOLDER = "Raw_Flow_Darkening_v2_3laneAverage18Lanes"

OUTPUT_ROOT = BASE_FOLDER / "Raw_Flow_7Lanes"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ----- Choose which Stage-2 export to use -----
EXPORT_MODE = "aic_45s_1vps"
STRICT_ALL_FOLDERS = False

# ===== Exclusions (old data or data that was recollected) =====
EXCLUDE_GH_FAMILY_FOLDERS = True
EXCLUDE_EXPERIMENT_BASE_KEYS = {
    "10.15.25.JR.PBSRedoCaffeine.3",
}
EXCLUDE_EXPERIMENT_FOLDERS_EXACT = set()

# ML cleaning filters
KEEP_CHANNELS = {1, 2, 3, 4}
EXCLUDE_BACTERIA = {"D"}
DROP_DI_GUARD = True

# Duplicate reporting behavior
WRITE_DUP_REPORT = False
SHOW_DUP_GROUPS = 10
AUTO_RESOLVE_DUPLICATES = True
REDO_TOKEN_REGEX = re.compile(r"redo", flags=re.IGNORECASE)
REDO_CONDITION_COLS = ("Combo", "ChipNumber")

# ========================= Helpers: exclusion logic=========================
def _strip_part_suffix(exp_name: str) -> str:
    return re.sub(r'\.(?:p|pt)\d+$', '', str(exp_name), flags=re.IGNORECASE)

def _is_gh_family_folder(exp_name: str) -> bool:
    toks = [t.upper() for t in str(exp_name).split(".") if t]
    return any(t in ("GH", "GHD", "GDH") for t in toks)

def discover_valid_experiment_folders(base_dir: Path):
    base_dir = Path(base_dir)
    found = []

    for reagent_dir in sorted([p for p in base_dir.iterdir() if p.is_dir()]):
        if reagent_dir.name.startswith("."):
            continue
        if reagent_dir.name in {"Raw_Flow_7Lanes", "Stage2_Global", "Stage2_PerReagent", "Stage2_Scratch"}:
            continue

        for exp_dir in sorted([p for p in reagent_dir.iterdir() if p.is_dir()]):
            raw_csv_dir = exp_dir / "Data" / OUTPUT_SUBFOLDER / "Raw_CSVs"
            if raw_csv_dir.exists() and raw_csv_dir.is_dir():
                found.append(exp_dir)

    print(f"Found {len(found)} experiment folder(s) with {OUTPUT_SUBFOLDER}/Raw_CSVs under: {base_dir}")

    kept = []
    dropped_gh = dropped_base = dropped_exact = 0

    for exp_dir in found:
        exp_name = exp_dir.name

        if exp_name in EXCLUDE_EXPERIMENT_FOLDERS_EXACT:
            dropped_exact += 1
            continue

        if EXCLUDE_GH_FAMILY_FOLDERS and _is_gh_family_folder(exp_name):
            dropped_gh += 1
            continue

        base_key = _strip_part_suffix(exp_name)
        if base_key in set(EXCLUDE_EXPERIMENT_BASE_KEYS):
            dropped_base += 1
            continue

        kept.append(exp_dir)

    if dropped_exact:
        print(f" Excluded {dropped_exact} folder(s) due to EXCLUDE_EXPERIMENT_FOLDERS_EXACT.")
    if dropped_gh:
        print(f" Excluded {dropped_gh} folder(s) due to GH/GHD/GDH token rule.")
    if dropped_base:
        print(f" Excluded {dropped_base} folder(s) due to explicit base-key exclusions.")
    print(f" After exclusions: {len(kept)} experiment folder(s) remain.")
    return kept

# ========================= Parsing / cleaning =========================
def parse_channel_lane(chan: str):
    if not isinstance(chan, str):
        return (None, None)
    s = chan.strip()
    m = re.match(r'^(?:C|Ch)\s*(\d+)\s*(?:[._-]*P\s*(\d+))?$', s, flags=re.IGNORECASE)
    if not m:
        return (None, None)
    ch = int(m.group(1))
    ln = int(m.group(2)) if m.group(2) else 1
    return (ch, ln)

def parse_chip_from_experiment_folder(exp_folder: str):
    s = str(exp_folder)
    s = re.sub(r'\.(?:p|pt)\d+$', '', s, flags=re.IGNORECASE)
    toks = [t for t in s.split(".") if t != ""]
    for t in reversed(toks):
        if t.isdigit():
            return int(t)
    return None

def is_di_row(row):
    combo = str(row.get("Combo", ""))
    plastic = str(row.get("Plastic", "")).upper()
    reagent = str(row.get("Reagent", "")).upper()
    if re.search(r'PD\.?\s*QN', combo, flags=re.IGNORECASE):
        return True
    if plastic == "D" and reagent == "N":
        return True
    return False

def safe_read_csv(path: Path):
    try:
        if not path.exists():
            return None
        if path.stat().st_size == 0:
            print(f"[WARN] empty file (0 bytes): {path}")
            return None
        df = pd.read_csv(path)
        if df is None or len(df.columns) == 0 or df.shape[0] == 0:
            print(f"[WARN] empty/unreadable: {path}")
            return None
        return df
    except Exception as e:
        print(f"[WARN] failed reading {path.name}: {e}")
        return None

# ========================= Export selection =========================
EXPORT_SUFFIX_MAP = {
    "aic_10s_1vps": "_Raw_Average_aic_10s_1vps.csv",
    "aic_45s_1vps": "_Raw_Average_aic_45s_1vps.csv",
}

def detect_best_common_suffix(exp_folders):

    suffix_counts = {}
    example_path = {}

    for exp_dir in exp_folders:
        exp_name = exp_dir.name
        for f in exp_dir.glob(f"{exp_name}_Raw_Average_aic*.csv"):
            suf = f.name.replace(exp_name, "", 1)  # keep the suffix portion
            suffix_counts[suf] = suffix_counts.get(suf, 0) + 1
            example_path.setdefault(suf, str(f))

    if not suffix_counts:
        return None, 0, None

    best_suf = max(suffix_counts, key=lambda k: suffix_counts[k])
    return best_suf, suffix_counts[best_suf], example_path.get(best_suf)

def experiment_folders_to_stage2_exports_paired(exp_folders):

    if EXPORT_MODE in EXPORT_SUFFIX_MAP:
        chosen_suffix = EXPORT_SUFFIX_MAP[EXPORT_MODE]
        mode_tag = EXPORT_MODE
        # check coverage
        have = 0
        for exp_dir in exp_folders:
            exp_name = exp_dir.name
            p = exp_dir / f"{exp_name}{chosen_suffix}"
            if p.exists():
                have += 1

        print(f"\n===  Using requested export mode: {EXPORT_MODE} ===")
        print(f"Suffix: {chosen_suffix}")
        print(f"Found in {have}/{len(exp_folders)} experiment folders")

        if STRICT_ALL_FOLDERS and have != len(exp_folders):
            raise FileNotFoundError(
                f"Requested export '{EXPORT_MODE}' not present in all folders: {have}/{len(exp_folders)}.\n"
                f"Either re-run Stage-2 to generate these exports everywhere, or set EXPORT_MODE='auto'."
            )

        pairs = []
        missing = 0
        for exp_dir in exp_folders:
            exp_name = exp_dir.name
            csv_path = exp_dir / f"{exp_name}{chosen_suffix}"
            reagent_name = exp_dir.parent.name
            if csv_path.exists():
                pairs.append((csv_path, reagent_name))
            else:
                missing += 1

        if missing:
            print(f"[WARN] {missing} folders missing requested export; they will be excluded from ML build.")
        print(f"📄 ML will load {len(pairs)} exported slope CSV(s) [{mode_tag}].")
        return pairs, f"Raw_Average_{mode_tag}"


    best_suf, best_n, example = detect_best_common_suffix(exp_folders)
    if best_suf is None:
        raise FileNotFoundError("No *_Raw_Average_aic*.csv exports found in any experiment folder.")

    print(f"\n===  AUTO detected Stage-2 export pattern ===")
    print(f"Suffix chosen: {best_suf}")
    print(f"Found in {best_n}/{len(exp_folders)} experiment folders")
    print(f"Example: {example}")

    pairs = []
    missing = 0
    for exp_dir in exp_folders:
        exp_name = exp_dir.name
        csv_path = exp_dir / f"{exp_name}{best_suf}"
        reagent_name = exp_dir.parent.name
        if csv_path.exists():
            pairs.append((csv_path, reagent_name))
        else:
            missing += 1

    if missing:
        print(f"[WARN] {missing} folders missing auto-detected export; excluded.")
    print(f" ML will load {len(pairs)} exported slope CSV(s) [AUTO].")
    tag = best_suf.replace(".csv", "").lstrip("_")
    return pairs, tag


def apply_redo_override_by_condition(dfr: pd.DataFrame,
                                    condition_cols=("Combo", "ChipNumber"),
                                    folder_col="Experiment Folder",
                                    redo_regex=REDO_TOKEN_REGEX,
                                    verbose=True,
                                    reagent_name=None):
    if folder_col not in dfr.columns:
        return dfr

    dfr = dfr.copy()
    dfr["_is_redo"] = dfr[folder_col].astype(str).apply(lambda s: bool(redo_regex.search(s)))

    has_redo = (
        dfr.groupby(list(condition_cols), dropna=False)["_is_redo"]
           .any()
           .reset_index()
           .rename(columns={"_is_redo": "_cond_has_redo"})
    )
    dfr = dfr.merge(has_redo, on=list(condition_cols), how="left")

    before = len(dfr)
    keep = (~dfr["_cond_has_redo"]) | (dfr["_cond_has_redo"] & dfr["_is_redo"])
    out = dfr.loc[keep].copy()
    out.drop(columns=["_is_redo", "_cond_has_redo"], inplace=True, errors="ignore")

    if verbose:
        n_cond_total = dfr[list(condition_cols)].drop_duplicates().shape[0]
        n_cond_overridden = int(has_redo["_cond_has_redo"].sum())
        dropped = before - len(out)
        tag = f"{reagent_name}" if reagent_name else "reagent"
        print(f" Redo override ({tag}): conditions total={n_cond_total} | overridden={n_cond_overridden} | rows dropped={dropped}")

    return out


def audit_and_optionally_resolve_duplicates(dfr: pd.DataFrame, key_cols, slope_col: str, reagent: str, tag: str):
    dup_mask = dfr.duplicated(subset=key_cols, keep=False)
    if not dup_mask.any():
        return dfr

    dup_df = dfr.loc[dup_mask].copy()
    cols_keep = key_cols + ["Experiment Folder", "SourceFile", "SourceFolder", slope_col, "WindowEdge"]
    cols_keep = [c for c in cols_keep if c in dup_df.columns]
    dup_df2 = dup_df[cols_keep].copy()

    grp = dup_df2.groupby(key_cols, dropna=False)
    summary = grp.agg(
        n_rows=(slope_col, "size"),
        folders=("Experiment Folder", lambda x: sorted(set(map(str, x)))) if "Experiment Folder" in dup_df2.columns else (slope_col, lambda x: []),
        files=("SourceFile", lambda x: sorted(set(map(str, x)))) if "SourceFile" in dup_df2.columns else (slope_col, lambda x: []),
    ).reset_index()

    print(f"\n DUPLICATES FOUND for reagent={reagent}")
    print(f"   Duplicate rows: {dup_df2.shape[0]}")
    print(f"   Duplicate pivot-cells (unique keys): {summary.shape[0]}")
    show = summary.sort_values(["n_rows"], ascending=False).head(SHOW_DUP_GROUPS)
    pd.set_option("display.max_colwidth", 160)
    print("\n--- Top duplicate groups ---")
    print(show.to_string(index=False))

    if WRITE_DUP_REPORT:
        out_summary = OUTPUT_ROOT / f"Duplicate_Report_{tag}_{reagent}.csv"
        out_rows = OUTPUT_ROOT / f"Duplicate_Rows_{tag}_{reagent}.csv"
        summary.to_csv(out_summary, index=False)
        dup_df2.to_csv(out_rows, index=False)
        print(f"\n wrote duplicate summary: {out_summary}")
        print(f" wrote duplicate rows   : {out_rows}")

    if not AUTO_RESOLVE_DUPLICATES:
        raise ValueError(f"Duplicate slope rows detected for reagent={reagent}.")

    if "Experiment Folder" not in dfr.columns:
        raise ValueError("Cannot auto-resolve duplicates: missing 'Experiment Folder' column.")

    dfr = dfr.copy()
    dfr["_redo_priority"] = dfr["Experiment Folder"].astype(str).apply(
        lambda s: 0 if REDO_TOKEN_REGEX.search(s) else 1
    )
    dfr_sorted = dfr.sort_values(by=key_cols + ["_redo_priority"], ascending=True, kind="mergesort")
    keep_mask = ~dfr_sorted.duplicated(subset=key_cols, keep="first")
    kept = dfr_sorted.loc[keep_mask].copy()
    kept.drop(columns=["_redo_priority"], inplace=True, errors="ignore")
    print(f"\n Auto-resolved duplicates for reagent={reagent} (prefer redo). Rows kept={len(kept)}")
    return kept

# ========================= ML build =========================
def build_ml_tables(stage2_pairs, tag):
    if not stage2_pairs:
        raise FileNotFoundError("No Stage-2 slope CSVs were discovered to build ML tables from.")

    dfs = []
    read_ok = read_bad = 0

    for (p, reagent_name) in stage2_pairs:
        df = safe_read_csv(p)
        if df is None:
            read_bad += 1
            continue

        df["SourceFile"] = p.name
        df["SourceFolder"] = str(p.parent)
        df["ReagentName_Folder"] = reagent_name

        if "Experiment Folder" not in df.columns:
            df["Experiment Folder"] = p.parent.name

        dfs.append(df)
        read_ok += 1

    print(f" Read OK: {read_ok} | Skipped (empty/bad): {read_bad}")
    if not dfs:
        raise RuntimeError("All discovered Stage-2 slope CSVs were empty/unreadable.")

    long_df = pd.concat(dfs, ignore_index=True)
    print(f" Loaded rows (long): {len(long_df)}")

    base_required = {"Combo", "Channel", "WindowEdge", "Experiment Folder"}
    missing = [c for c in base_required if c not in long_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in slope CSVs: {missing}")

    if "Normalized_Slope" in long_df.columns:
        slope_col = "Normalized_Slope"
    elif "Slope" in long_df.columns:
        slope_col = "Slope"
        print("[WARN] Normalized_Slope missing -> using raw Slope.")
    else:
        raise ValueError("Neither 'Normalized_Slope' nor 'Slope' exists in the exported CSVs.")

    ch_lane = long_df["Channel"].apply(lambda s: pd.Series(parse_channel_lane(str(s))))
    ch_lane.columns = ["ChannelNum", "Lane"]
    long_df = pd.concat([long_df, ch_lane], axis=1)

    before = len(long_df)
    long_df = long_df.dropna(subset=["ChannelNum", "Lane"])
    long_df["ChannelNum"] = long_df["ChannelNum"].astype(int)
    long_df["Lane"] = long_df["Lane"].astype(int)
    print(f"    Channel/Lane parsed. Rows {before}->{len(long_df)}")

    before = len(long_df)
    long_df = long_df[long_df["ChannelNum"].isin(KEEP_CHANNELS)].copy()
    print(f" channel filter (kept {sorted(KEEP_CHANNELS)}): {before} → {len(long_df)}")

    if EXCLUDE_BACTERIA:
        if "Plastic" in long_df.columns:
            before = len(long_df)
            long_df = long_df[~long_df["Plastic"].astype(str).str.upper().isin(EXCLUDE_BACTERIA)].copy()
            print(f" dropped bacteria {sorted(EXCLUDE_BACTERIA)} (Plastic): {before} → {len(long_df)}")

    if DROP_DI_GUARD:
        before = len(long_df)
        long_df = long_df[~long_df.apply(is_di_row, axis=1)].copy()
        print(f" dropped DI rows (guard): {before} → {len(long_df)}")

    before = len(long_df)
    long_df["WindowEdge"] = pd.to_numeric(long_df["WindowEdge"], errors="coerce")
    long_df[slope_col] = pd.to_numeric(long_df[slope_col], errors="coerce")
    long_df = long_df.dropna(subset=["WindowEdge", slope_col])
    print(f" numeric keep (WindowEdge + {slope_col}): {before} → {len(long_df)}")

    before = len(long_df)
    chips = long_df["Experiment Folder"].apply(parse_chip_from_experiment_folder)
    long_df["ChipNumber"] = pd.to_numeric(chips, errors="coerce")
    long_df = long_df.dropna(subset=["ChipNumber"])
    long_df["ChipNumber"] = long_df["ChipNumber"].astype(int)
    print(f" chip parsed keep: {before} → {len(long_df)}")
    print(f"   ↳ Chips seen: {sorted(long_df['ChipNumber'].unique().tolist())}")

    long_df["ReagentName"] = long_df["ReagentName_Folder"].astype(str).str.strip()
    print("\nReagents (from folder truth):")
    print("  ", sorted(long_df["ReagentName"].unique().tolist()))

    INDEX_COLS = ["Combo", "ChipNumber", "ChannelNum", "Lane"]

    print("\nCoverage by reagent (pre-pivot):")
    for r, g in long_df.groupby("ReagentName"):
        npos = g[INDEX_COLS].drop_duplicates().shape[0]
        print(f"   • {r:<10}: {npos:6d} positions")

    pivots = []
    for reagent in sorted(long_df["ReagentName"].unique().tolist()):
        dfr = long_df[long_df["ReagentName"] == reagent].copy()
        if dfr.empty:
            continue

        dfr = apply_redo_override_by_condition(
            dfr,
            condition_cols=REDO_CONDITION_COLS,
            folder_col="Experiment Folder",
            verbose=True,
            reagent_name=reagent
        )

        dfr["FeatureLabel"] = dfr["WindowEdge"].astype(int).map(lambda w: f"{reagent}_{w}")
        key_cols = INDEX_COLS + ["FeatureLabel"]

        dfr = audit_and_optionally_resolve_duplicates(dfr, key_cols, slope_col, reagent, tag)

        pivot = (
            dfr.set_index(INDEX_COLS + ["FeatureLabel"])[slope_col]
               .unstack("FeatureLabel")
               .reset_index()
        )
        pivots.append(pivot)
        print(f"    {reagent}: pivot shape {pivot.shape}")

    aligned = reduce(lambda L, R: pd.merge(L, R, on=INDEX_COLS, how="inner"), pivots)
    feature_cols = [c for c in aligned.columns if c not in INDEX_COLS]
    print("\nML Aligned Table ")
    print(f"Rows: {aligned.shape[0]}  |  Features: {len(feature_cols)}")

    return long_df, aligned

def main():
    valid_exp_folders = discover_valid_experiment_folders(BASE_FOLDER)
    stage2_pairs, tag = experiment_folders_to_stage2_exports_paired(valid_exp_folders)

    long_df, pivot_df = build_ml_tables(stage2_pairs, tag=tag)

    long_out  = OUTPUT_ROOT / f"ML_Long_{tag}__031326.csv"
    pivot_out = OUTPUT_ROOT / f"ML_Aligned_Complete_Pivot_Table_{tag}_031326.csv"

    long_df.to_csv(long_out, index=False)
    pivot_df.to_csv(pivot_out, index=False)

    print(f"\n wrote {long_out}")
    print(f" wrote {pivot_out}")

if __name__ == "__main__":
    main()


#**ML Models**

##**Bacteria Species**

In [ ]:
'''#!/usr/bin/env python
# coding: utf-8
BACTERIA SPECIES —  Group-aware LOOCV + Top-N + Tuning

CLASS_MODE = "combo"   -> classify Combo labels
CLASS_MODE = "species" -> classify species labels only

 1) Baseline LOOCV on all features
 2) Top-N scan using GroupKFold over N = 1..360 (no skipping)
 3) LOOCV on best Top-N feature subset
 4) GroupKFold grid search on Top-N features
 5) LOOCV on tuned model
 6) Fit final tuned model on all data and save artifact

 PRIMARY MODEL-SELECTION METRIC
 - Accuracy
 - Macro F1

 GROUPING
 - LeaveOneGroupOut / GroupKFold groups are:
     Combo + ChipNumber + ChannelNum
'''
import os
import json
import time
import warnings
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    ParameterGrid,
    cross_val_predict,
    cross_val_score,
    learning_curve,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

FORCE_RERUN = False
GK_N_SPLITS = 5
SEED_LIST = list(range(1, 1))
TOPN_MAX = 360

TASK_NAME = "Bacteria Species"
TASK_TYPE = "multiclass"
PRIMARY_SCORING = "accuracy"

# "combo" or "species"
CLASS_MODE = "species"

BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/Bacteria Species In PBS")
#wrote /groups/jyyoon/OneHealth Skin Microbiome/Bacteria Species In PBS/Raw_Flow_7Lanes/ML_Aligned_Complete_Pivot_Table_Raw_Average_aic_45s_1vps_031326.csv
DATA_FILE = BASE_FOLDER / "Raw_Flow_7Lanes" / "ML_Aligned_Complete_Pivot_Table_Raw_Average_aic_45s_1vps_031326.csv"
RESULTS_DIR = BASE_FOLDER / "ML_Results_Species_Unified_360TopN"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CMAPS = {
    "Random Forest": plt.get_cmap("Purples"),
    "SVM": plt.get_cmap("Greens"),
    "Logistic Regression": plt.get_cmap("Oranges"),
    "LDA": plt.get_cmap("YlOrBr"),
    "XGBoost": plt.get_cmap("Blues"),
}


GRID = {
    "Random Forest": {
        "n_estimators": [200, 400, 800],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced"],
    },

    "SVM": [
        {
            "kernel": ["rbf"],
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", 0.01, 0.001],
            "class_weight": [None, "balanced"],
        },
        {
            "kernel": ["linear"],
            "C": [0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
        },
    ],

    "Logistic Regression": [
        {
            "solver": ["lbfgs"],
            "penalty": ["l2"],
            "C": [0.01, 0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
            "multi_class": ["multinomial"],
        },
        {
            "solver": ["saga"],
            "penalty": ["l1", "l2", "elasticnet"],
            "C": [0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0.0, 0.5, 1.0],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
            "multi_class": ["multinomial"],
        },
    ],

    "LDA": [
        {"solver": ["svd"]},
        {"solver": ["lsqr", "eigen"], "shrinkage": ["auto", 0.1, 0.5]},
    ],
}

if HAS_XGB:
    GRID["XGBoost"] = {
        "n_estimators": [100, 200, 400],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }



def make_models_for_seed(seed: int):
    models = {
        "Random Forest": RandomForestClassifier(random_state=seed),
        "SVM": SVC(kernel="rbf", probability=True, random_state=seed),
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            multi_class="multinomial",
            random_state=seed
        ),
        "LDA": LinearDiscriminantAnalysis(),
    }
    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            eval_metric="mlogloss",
            random_state=seed
        )
    return models


def add_text_page(pdf, title, text, fontsize=9):
    fig = plt.figure(figsize=(8.5, 11))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    fig.text(0.5, 0.97, title, ha="center", va="top", fontsize=11, weight="bold")
    fig.text(0.02, 0.93, text, ha="left", va="top", fontsize=fontsize, family="monospace")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

def save_metrics_row_csv(row_dict, out_csv):
    out_csv = Path(out_csv)
    df = pd.DataFrame([row_dict])
    df.to_csv(out_csv, mode="a", index=False, header=not out_csv.exists())

def plot_learning_curve_cv(estimator, X, y, groups, model_name, stage, pdf,
                           cv=None, n_points=10, rng_seed=42):
    if cv is None:
        cv = GroupKFold(n_splits=GK_N_SPLITS)

    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(estimator))])

    fig, ax = plt.subplots(figsize=(7, 5))
    train_sizes, train_scores, test_scores = learning_curve(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, n_points),
        shuffle=True,
        random_state=rng_seed,
    )
    tr_mean, tr_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
    te_mean, te_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)

    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)
    ax.fill_between(train_sizes, te_mean - te_std, te_mean + te_std, alpha=0.1)
    ax.plot(train_sizes, tr_mean, "o-", label="Training score")
    ax.plot(train_sizes, te_mean, "o-", label="CV score")
    ax.set_title(f"{model_name} — {stage} Learning Curve")
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Accuracy")
    ax.grid(True)
    ax.legend(loc="best")

    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_curve(scores_by_n, model_name, metric_name, pdf):
    fig = plt.figure()
    n_vals, sc_vals = zip(*scores_by_n)
    plt.plot(n_vals, sc_vals, marker="o")
    plt.title(f"{model_name} - {metric_name} vs Top N Features (GroupKFold)")
    plt.xlabel("Top N Features")
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_importance(feat_df, model_name, pdf, top_n=20):
    feat_df_sorted = feat_df.sort_values("Importance", ascending=False).head(top_n)
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_df_sorted, x="Importance", y="Feature")
    plt.title(f"{model_name} - Top {top_n} Feature Importances")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)


def normalize_combo_string(x: str) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x).strip()
    s = s.replace(",", ".").replace(";", ".").replace("|", ".").replace("/", ".")
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"\.+", ".", s)
    s = s.strip(".")
    return s

# direct species format like "C.QM"
SPECIES_FROM_COMBO_REGEX = True

# map from prefix letter after "P"
SPECIES_MAP_BY_PREFIX = {
     "C": "C",
     "I": "E",
     "S": "A",
     "T": "S",
     "Z": "H",
}

def derive_species_from_combo(combo: str) -> str:
    if combo is None:
        return None

    s = normalize_combo_string(combo)
    if not s:
        return None

    if SPECIES_FROM_COMBO_REGEX:
        m = pd.Series([s]).str.extract(r"^([CISTZ])\.", expand=True).iloc[0, 0]
        if isinstance(m, str) and m:
            return m

    m2 = pd.Series([s]).str.extract(r"^P([A-Za-z]+)\.", expand=True).iloc[0, 0]
    if isinstance(m2, str) and m2:
        key = m2[0].upper()
        if SPECIES_MAP_BY_PREFIX:
            return SPECIES_MAP_BY_PREFIX.get(key)
        return key

    return None

def load_pivot_table_species(data_file: Path, class_mode: str) -> pd.DataFrame:
    df = pd.read_csv(data_file)

    required = {"Combo", "ChipNumber", "ChannelNum", "Lane"}
    missing_req = [c for c in required if c not in df.columns]
    if missing_req:
        raise ValueError(
            f"DATA_FILE is missing required columns {missing_req}. "
            f"Got columns: {sorted(df.columns.tolist())[:80]}..."
        )

    df["Combo"] = df["Combo"].apply(normalize_combo_string)
    df["ChipNumber"] = pd.to_numeric(df["ChipNumber"], errors="coerce")
    df["ChannelNum"] = pd.to_numeric(df["ChannelNum"], errors="coerce")
    df["Lane"] = pd.to_numeric(df["Lane"], errors="coerce")

    df = df.dropna(subset=["ChipNumber", "ChannelNum", "Lane"]).copy()
    df["ChipNumber"] = df["ChipNumber"].astype(int)
    df["ChannelNum"] = df["ChannelNum"].astype(int)
    df["Lane"] = df["Lane"].astype(int)

    df["GroupLabel"] = (
        df["Combo"]
        + "_Chip" + df["ChipNumber"].astype(str)
        + "_Chan" + df["ChannelNum"].astype(str)
    )

    if class_mode.lower() == "combo":
        df["Label"] = df["Combo"]

    elif class_mode.lower() == "species":
        df["Label"] = df["Combo"].apply(derive_species_from_combo)

        bad = df["Label"].isna().sum()
        if bad > 0:
            ex = (
                df.loc[df["Label"].isna(), "Combo"]
                .astype(str)
                .drop_duplicates()
                .head(20)
                .tolist()
            )
            raise ValueError(
                f"Could not derive species for {bad} rows. "
                f"Example Combo values:\n{ex}\n\n"
                f"Fix derive_species_from_combo() or SPECIES_MAP_BY_PREFIX."
            )
    else:
        raise ValueError("CLASS_MODE must be 'combo' or 'species'")

    return df

META_COLS = ["Combo", "Label", "ChipNumber", "ChannelNum", "Lane", "GroupLabel"]

def make_X(df: pd.DataFrame) -> pd.DataFrame:
    X = (
        df.drop(columns=[c for c in META_COLS if c in df.columns], errors="ignore")
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return X


def rank_features_generic(model, X, y, rng_seed):
    try:
        m = clone(model)
        if hasattr(m, "random_state"):
            try:
                m.set_params(random_state=rng_seed)
            except Exception:
                pass
        m.fit(X, y)

        if hasattr(m, "feature_importances_"):
            imp = np.asarray(m.feature_importances_)
        elif hasattr(m, "coef_"):
            coef = np.asarray(m.coef_)
            if coef.ndim == 1:
                imp = np.abs(coef)
            else:
                imp = np.abs(coef).mean(axis=0)
        else:
            raise AttributeError("No feature importance or coefficients.")
    except Exception:
        lr = LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            multi_class="multinomial",
            random_state=rng_seed
        )
        lr.fit(X, y)
        coef = np.asarray(lr.coef_)
        if coef.ndim == 1:
            imp = np.abs(coef)
        else:
            imp = np.abs(coef).mean(axis=0)

    imp = np.ravel(imp)[:X.shape[1]]

    return (
        pd.DataFrame({"Feature": X.columns, "Importance": imp})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )


def compute_metrics(y_true, y_pred, task_type="multiclass"):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "MacroF1": f1_score(y_true, y_pred, average="macro"),
    }

def resumable_topn_scan(
    model_name,
    base_model,
    X,
    y,
    groups,
    sorted_features,
    gk_splits,
    results_dir,
    rng_seed,
    primary_scoring="accuracy",
    task_type="multiclass",
):
    progress_csv = Path(results_dir) / f"TopFeatures_{model_name}_Train.csv"
    rows = []
    tried_n = set()

    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if not df_done.empty and "Top_N" in df_done.columns:
            rows = df_done.to_dict(orient="records")
            tried_n = set(df_done["Top_N"].astype(int).tolist())
            print(f"🔄 Resuming Top-N scan for {model_name}: {len(tried_n)} already done.")

    gkf = GroupKFold(n_splits=gk_splits)
    max_n = min(TOPN_MAX, len(sorted_features), X.shape[1])

    for n in range(1, max_n + 1):
        if n in tried_n:
            continue

        top_feats = sorted_features[:n]
        pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(base_model))])

        if hasattr(pipe.named_steps["est"], "random_state"):
            try:
                pipe.named_steps["est"].set_params(random_state=rng_seed)
            except Exception:
                pass

        preds = cross_val_predict(pipe, X[top_feats], y, cv=gkf, groups=groups, n_jobs=-1)
        metric_dict = compute_metrics(y, preds, task_type=task_type)

        primary_value = metric_dict["Accuracy"] if primary_scoring == "accuracy" else metric_dict["MacroF1"]

        row = {
            "Top_N": int(n),
            "Primary": float(primary_value),
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
        }

        rows.append(row)
        pd.DataFrame(rows).to_csv(progress_csv, index=False)

        print(
            f"[{model_name}] Top {n} -> primary={row['Primary']:.4f} | "
            f"acc={row['Accuracy']:.4f} | macroF1={row['MacroF1']:.4f}"
        )

    return pd.DataFrame(rows)

def resumable_group_grid_search(
    model_name,
    base_estimator,
    param_grid,
    X,
    y,
    groups,
    results_dir,
    scoring="accuracy",
    cv_splits=5,
    random_state=42,
    tag="",
):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    progress_csv = results_dir / f"GridSearch_Resume_{model_name}{tag}.csv"
    best_params_json = results_dir / f"BestParams_{model_name}{tag}.json"

    combos = list(ParameterGrid(param_grid)) if param_grid else [{}]
    gkf = GroupKFold(n_splits=cv_splits)

    tried = set()
    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if "params_json" in df_done.columns:
            tried = set(df_done["params_json"].astype(str).tolist())

    best_score, best_params = -np.inf, None
    if best_params_json.exists():
        try:
            prev = json.load(open(best_params_json))
            best_params = prev.get("best_params")
            best_score = prev.get("best_score", -np.inf)
        except Exception:
            pass

    print(f"🔎 {model_name}: {len(combos)} combos total; {len(tried)} already done.")

    for i, params in enumerate(combos, 1):
        key = json.dumps(params, sort_keys=True)
        if key in tried:
            continue

        est = clone(base_estimator)
        if hasattr(est, "random_state"):
            try:
                est.set_params(random_state=random_state)
            except Exception:
                pass

        try:
            est.set_params(**params)
        except ValueError as e:
            pd.DataFrame([{
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model": model_name,
                "params_json": key,
                "mean_score": np.nan,
                "std_score": np.nan,
                "n_folds": 0,
                "note": f"incompatible: {e}",
            }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())
            continue

        pipe = Pipeline([("scaler", StandardScaler()), ("est", est)])
        scores = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring=scoring, n_jobs=-1)
        mean_score, std_score = float(np.mean(scores)), float(np.std(scores))

        pd.DataFrame([{
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "model": model_name,
            "params_json": key,
            "mean_score": mean_score,
            "std_score": std_score,
            "n_folds": len(scores),
        }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())

        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            json.dump({"best_params": best_params, "best_score": best_score}, open(best_params_json, "w"))

        print(
            f"[{model_name}] {i}/{len(combos)} params={params} -> "
            f"{scoring}={mean_score:.4f} (±{std_score:.4f}) | best={best_score:.4f}"
        )

    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("est", clone(base_estimator).set_params(**(best_params or {})))
    ])

    return best_pipe, best_params, best_score

def evaluate_stage_loocv(
    model,
    X,
    y,
    groups,
    model_name,
    stage,
    cmap,
    labels,
    pdf,
    results_dir,
    seed,
    data_file_name,
    task_type="multiclass",
):
    logo = LeaveOneGroupOut()
    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(model))])

    if hasattr(pipe.named_steps["est"], "random_state"):
        try:
            pipe.named_steps["est"].set_params(random_state=seed)
        except Exception:
            pass

    preds = cross_val_predict(pipe, X, y, cv=logo, groups=groups, n_jobs=-1)
    metric_dict = compute_metrics(y, preds, task_type=task_type)

    report_str = classification_report(y, preds, target_names=labels)

    print(
        f"\n{model_name} - {stage} LOOCV "
        f"| Acc={metric_dict['Accuracy']:.3f} "
        f"| MacroF1={metric_dict['MacroF1']:.3f}"
    )

    add_text_page(pdf, f"{model_name} — {stage} LOOCV (seed {seed})", report_str)

    fig_cm = plt.figure()
    sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d",
                cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(f"{model_name} - {stage} Confusion Matrix (LOOCV, seed {seed})")
    plt.tight_layout()
    pdf.savefig(fig_cm)
    plt.show()
    plt.close(fig_cm)

    try:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="predict_proba", n_jobs=-1)
    except Exception:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="decision_function", n_jobs=-1)
        y_scores = np.asarray(y_scores)
        if y_scores.ndim == 1:
            y_scores = y_scores.reshape(-1, 1)

    class_ids = np.arange(len(labels))
    Y = label_binarize(y, classes=class_ids)
    S = np.asarray(y_scores)

    if S.ndim == 1:
        S = S.reshape(-1, 1)

    if S.max() > 1.0 or S.min() < 0.0:
        S = (S - S.min(axis=0)) / (S.max(axis=0) - S.min(axis=0) + 1e-9)

    fig_roc, ax_roc = plt.subplots()
    fig_pr, ax_pr = plt.subplots()

    for i, label in enumerate(labels):
        fpr, tpr, _ = roc_curve(Y[:, i], S[:, i])
        prec, rec, _ = precision_recall_curve(Y[:, i], S[:, i])
        ax_roc.plot(fpr, tpr, label=f"{label} (AUC={auc(fpr, tpr):.2f})")
        ax_pr.plot(rec, prec, label=f"{label} (AUC={auc(rec, prec):.2f})")

    ax_roc.plot([0, 1], [0, 1], "k--")
    ax_roc.set_title(f"{model_name} - {stage} ROC (LOOCV, seed {seed})")
    ax_pr.set_title(f"{model_name} - {stage} PR (LOOCV, seed {seed})")
    ax_roc.set_xlabel("FPR")
    ax_roc.set_ylabel("TPR")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_roc.legend()
    ax_pr.legend()

    plt.tight_layout()
    pdf.savefig(fig_roc)
    pdf.savefig(fig_pr)
    plt.show()
    plt.close(fig_roc)
    plt.close(fig_pr)

    save_metrics_row_csv(
        {
            "Model": model_name,
            "Stage": stage,
            "Seed": seed,
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
            "DataFile": data_file_name,
        },
        Path(results_dir) / "All_Metrics_LOOCV.csv"
    )

    return metric_dict, preds

def print_group_summary(X, y, groups, labels=None, dataset_name="Dataset"):
    import pandas as pd
    import numpy as np

    groups_series = pd.Series(groups).astype(str)
    y_array = np.asarray(y)

    print("\n" + "=" * 70)
    print(f"{dataset_name} SUMMARY")
    print("=" * 70)
    print(f"Rows: {len(y_array):,}")
    print(f"Features: {X.shape[1]:,}")
    print(f"Unique groups: {groups_series.nunique():,}")

    group_sizes = groups_series.value_counts()
    print(f"Min rows per group: {group_sizes.min():,}")
    print(f"Max rows per group: {group_sizes.max():,}")
    print(f"Mean rows per group: {group_sizes.mean():.2f}")

    if labels is not None:
        print(f"Classes ({len(labels)}): {labels}")

    print("\nFirst 10 groups:")
    print(group_sizes.head(10).to_string())

    print("=" * 70 + "\n")

def evaluate_model_end_to_end(
    model_name,
    base_model,
    X,
    y,
    groups,
    labels,
    cmap,
    gk_splits,
    rng_seed,
    results_dir_seed,
    data_file_name,
    primary_scoring="accuracy",
    task_type="multiclass",
):
    pdf_path = Path(results_dir_seed) / f"{model_name}_Results_Seed{rng_seed}.pdf"
    summary = {
        "Model": model_name,
        "Seed": rng_seed,
        "Task": TASK_NAME,
        "TaskType": task_type,
        "PrimaryScoring": primary_scoring,
        "ClassMode": CLASS_MODE,
    }

    gkf_for_curves = GroupKFold(n_splits=gk_splits)

    with PdfPages(pdf_path) as pdf:

        baseline_metrics, _ = evaluate_stage_loocv(
            base_model, X, y, groups,
            model_name, "Baseline", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Baseline_LOOCV_Accuracy"] = baseline_metrics["Accuracy"]
        summary["Baseline_LOOCV_MacroF1"] = baseline_metrics["MacroF1"]

        plot_learning_curve_cv(
            base_model, X, y, groups, model_name, "Baseline", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        feat_df = rank_features_generic(base_model, X, y, rng_seed)
        sorted_names = feat_df["Feature"].tolist()[:min(TOPN_MAX, X.shape[1])]

        df_topn = resumable_topn_scan(
            model_name=model_name,
            base_model=base_model,
            X=X,
            y=y,
            groups=groups,
            sorted_features=sorted_names,
            gk_splits=gk_splits,
            results_dir=results_dir_seed,
            rng_seed=rng_seed,
            primary_scoring=primary_scoring,
            task_type=task_type,
        )

        best_row = df_topn.sort_values("Primary", ascending=False).iloc[0]
        best_n = int(best_row["Top_N"])
        top_features = sorted_names[:best_n]
        summary["TopN"] = best_n

        plot_feature_curve(
            list(zip(df_topn["Top_N"].tolist(), df_topn["Primary"].tolist())),
            model_name,
            f"Primary({primary_scoring})",
            pdf
        )
        plot_feature_importance(feat_df, model_name, pdf)

        pd.DataFrame({"Feature": top_features}).to_csv(
            Path(results_dir_seed) / f"TopFeatures_{model_name}_Selected.csv",
            index=False
        )

        top_metrics, _ = evaluate_stage_loocv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["TopFeatures_LOOCV_Accuracy"] = top_metrics["Accuracy"]
        summary["TopFeatures_LOOCV_MacroF1"] = top_metrics["MacroF1"]

        plot_learning_curve_cv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        _, best_params, _ = resumable_group_grid_search(
            model_name=model_name,
            base_estimator=base_model,
            param_grid=GRID.get(model_name, {}),
            X=X[top_features],
            y=y,
            groups=groups,
            results_dir=results_dir_seed,
            scoring=primary_scoring,
            cv_splits=gk_splits,
            random_state=rng_seed,
        )

        tuned_best = clone(base_model).set_params(**(best_params or {}))
        if hasattr(tuned_best, "random_state"):
            try:
                tuned_best.set_params(random_state=rng_seed)
            except Exception:
                pass

        tuned_metrics, _ = evaluate_stage_loocv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Tuned_LOOCV_Accuracy"] = tuned_metrics["Accuracy"]
        summary["Tuned_LOOCV_MacroF1"] = tuned_metrics["MacroF1"]
        summary["Best_Params"] = best_params

        plot_learning_curve_cv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        final_pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(tuned_best))])
        final_pipe.fit(X[top_features], y)

        artifact = {
            "pipeline": final_pipe,
            "top_features": top_features,
            "labels": labels,
            "best_params": best_params,
            "data_file": data_file_name,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "seed": rng_seed,
            "task": TASK_NAME,
            "primary_scoring": primary_scoring,
            "task_type": task_type,
            "class_mode": CLASS_MODE,
        }
        joblib.dump(artifact, Path(results_dir_seed) / f"{model_name}_Final_Fitted.joblib")

    return summary

# =================================== MAIN===================================
if __name__ == "__main__":
    df_all = load_pivot_table_species(DATA_FILE, CLASS_MODE)

    X = make_X(df_all)
    if X.shape[1] == 0:
        raise RuntimeError("No numeric feature columns found after dropping metadata columns.")

    le = LabelEncoder()
    labels_all = sorted(df_all["Label"].unique())
    le.fit(labels_all)
    y = le.transform(df_all["Label"])
    labels = list(le.classes_)
    groups = df_all["GroupLabel"].astype(str).values

    print_group_summary(
    X=X,
    y=y,
    groups=groups,
    labels=labels,
    dataset_name=f"Bacteria Species ({CLASS_MODE})"
    )

    print(f" DATA_FILE = {DATA_FILE}")
    print(f" Rows = {len(df_all):,}")
    print(f" Groups = {pd.Series(groups).nunique():,}")
    print(f" ClassMode = {CLASS_MODE}")
    print(f" Labels ({len(labels)}) = {labels}")
    print(f" Features = {X.shape[1]}")
    print(f" Primary scoring = {PRIMARY_SCORING}")
    print(f" XGBoost available = {HAS_XGB}")

    all_summaries = []

    for seed in SEED_LIST:
        print(f"\n SEED {seed}")
        results_dir_seed = RESULTS_DIR / f"Seed_{seed}"
        results_dir_seed.mkdir(parents=True, exist_ok=True)

        MODELS = make_models_for_seed(seed)
        per_seed_summaries = []

        for model_name, model in MODELS.items():
            print(f"\n Seed {seed} — Running {model_name}")

            cmap = MODEL_CMAPS.get(model_name, plt.get_cmap("Blues"))

            summary = evaluate_model_end_to_end(
                model_name=model_name,
                base_model=model,
                X=X,
                y=y,
                groups=groups,
                labels=labels,
                cmap=cmap,
                gk_splits=GK_N_SPLITS,
                rng_seed=seed,
                results_dir_seed=results_dir_seed,
                data_file_name=Path(DATA_FILE).name,
                primary_scoring=PRIMARY_SCORING,
                task_type=TASK_TYPE,
            )
            per_seed_summaries.append(summary)
            all_summaries.append(summary)

        per_seed_df = pd.DataFrame(per_seed_summaries)
        col_order = [
            "Model",
            "Seed",
            "Task",
            "TaskType",
            "PrimaryScoring",
            "ClassMode",
            "Baseline_LOOCV_Accuracy",
            "Baseline_LOOCV_MacroF1",
            "TopN",
            "TopFeatures_LOOCV_Accuracy",
            "TopFeatures_LOOCV_MacroF1",
            "Tuned_LOOCV_Accuracy",
            "Tuned_LOOCV_MacroF1",
            "Best_Params",
        ]
        cols_present = [c for c in col_order if c in per_seed_df.columns]
        per_seed_df = per_seed_df.loc[:, cols_present + [c for c in per_seed_df.columns if c not in cols_present]]

        per_seed_path = results_dir_seed / "Model_Comparison_Summary_Multiclass_Train.csv"
        per_seed_df.to_csv(per_seed_path, index=False)
        print(f" Seed {seed} summary saved → {per_seed_path}")

    summary_df = pd.DataFrame(all_summaries)
    col_order = [
        "Model",
        "Seed",
        "Task",
        "TaskType",
        "PrimaryScoring",
        "ClassMode",
        "Baseline_LOOCV_Accuracy",
        "Baseline_LOOCV_MacroF1",
        "TopN",
        "TopFeatures_LOOCV_Accuracy",
        "TopFeatures_LOOCV_MacroF1",
        "Tuned_LOOCV_Accuracy",
        "Tuned_LOOCV_MacroF1",
        "Best_Params",
    ]
    cols_present = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df.loc[:, cols_present + [c for c in summary_df.columns if c not in cols_present]]

    summary_path = RESULTS_DIR / "Model_Comparison_Summary_Multiclass_AllSeeds.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\n DONE.")

##**Bacteria Mixtures Multiclass**

In [ ]:
'''#!/usr/bin/env python
 coding: utf-8
 BACTERIA MIXTURES (Multiclass) — Unified Group-aware LOOCV + Top-N + Tuning

 TASK
 - Multiclass classification for mixtures dataset
 - Supports:
     CLASS_MODE = "combo"   -> 12-way combo classification (e.g., PK.QH, PE.QM, ...)
     CLASS_MODE = "species" -> 4-way species-only classification (E/F/K/R)

 STANDARDIZED PIPELINE
 1) Baseline LOOCV on all features
 2) Top-N scan using GroupKFold over N = 1..360 (no skipping)
 3) LOOCV on best Top-N feature subset
 4) GroupKFold grid search on Top-N features
 5) LOOCV on tuned model
 6) Fit final tuned model on all data and save artifact

 PRIMARY MODEL-SELECTION METRIC
 - Accuracy

 ALSO RECORDED
 - Accuracy
 - Macro F1

 GROUPING
 - LeaveOneGroupOut / GroupKFold groups are:
     Combo + ChipNumber + ChannelNum

'''
import os
import json
import time
import warnings
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    ParameterGrid,
    cross_val_predict,
    cross_val_score,
    learning_curve,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

FORCE_RERUN = False
GK_N_SPLITS = 5
SEED_LIST = list(range(1, 3))
TOPN_MAX = 360

TASK_NAME = "Bacteria Mixtures Multiclass"
TASK_TYPE = "multiclass"
PRIMARY_SCORING = "accuracy"

# "combo"   -> 12-way classification
# "species" -> 4-way classification (E/F/K/R)
CLASS_MODE = "species"

BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/(Redo) Bacterial Mixtures in PBS")
DATA_FILE = BASE_FOLDER / "ML_Tables" / "ML_Aligned_Complete_Pivot_Table_Mixtures_withoutNaCl_aic_45s_1vps_031922.csv"
RESULTS_DIR = BASE_FOLDER / "ML_Results" / "Mixtures_GroupLOOCV_Multiclass_Unified_360TopN_032026"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CMAPS = {
    "Random Forest": plt.get_cmap("Purples"),
    "SVM": plt.get_cmap("Greens"),
    "Logistic Regression": plt.get_cmap("Oranges"),
    "LDA": plt.get_cmap("YlOrBr"),
    "XGBoost": plt.get_cmap("Blues"),
}



GRID = {
    "Random Forest": {
        "n_estimators": [200, 400, 800],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced"],
    },

    "SVM": [
        {
            "kernel": ["rbf"],
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", 0.01, 0.001],
            "class_weight": [None, "balanced"],
        },
        {
            "kernel": ["linear"],
            "C": [0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
        },
    ],

    "Logistic Regression": [
        {
            "solver": ["lbfgs"],
            "penalty": ["l2"],
            "C": [0.01, 0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
            "multi_class": ["multinomial"],
        },
        {
            "solver": ["saga"],
            "penalty": ["l1", "l2", "elasticnet"],
            "C": [0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0.0, 0.5, 1.0],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
            "multi_class": ["multinomial"],
        },
    ],

    "LDA": [
        {"solver": ["svd"]},
        {"solver": ["lsqr", "eigen"], "shrinkage": ["auto", 0.1, 0.5]},
    ],
}

if HAS_XGB:
    GRID["XGBoost"] = {
        "n_estimators": [100, 200, 400],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }



def make_models_for_seed(seed: int):
    models = {
        "Random Forest": RandomForestClassifier(random_state=seed),
        "SVM": SVC(kernel="rbf", probability=True, random_state=seed),
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            multi_class="multinomial",
            random_state=seed
        ),
        "LDA": LinearDiscriminantAnalysis(),
    }
    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            eval_metric="mlogloss",
            random_state=seed
        )
    return models



def add_text_page(pdf, title, text, fontsize=9):
    fig = plt.figure(figsize=(8.5, 11))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    fig.text(0.5, 0.97, title, ha="center", va="top", fontsize=11, weight="bold")
    fig.text(0.02, 0.93, text, ha="left", va="top", fontsize=fontsize, family="monospace")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

def save_metrics_row_csv(row_dict, out_csv):
    out_csv = Path(out_csv)
    df = pd.DataFrame([row_dict])
    df.to_csv(out_csv, mode="a", index=False, header=not out_csv.exists())

def plot_learning_curve_cv(estimator, X, y, groups, model_name, stage, pdf,
                           cv=None, n_points=10, rng_seed=42):
    if cv is None:
        cv = GroupKFold(n_splits=GK_N_SPLITS)

    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(estimator))])

    fig, ax = plt.subplots(figsize=(7, 5))
    train_sizes, train_scores, test_scores = learning_curve(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, n_points),
        shuffle=True,
        random_state=rng_seed,
    )
    tr_mean, tr_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
    te_mean, te_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)

    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)
    ax.fill_between(train_sizes, te_mean - te_std, te_mean + te_std, alpha=0.1)
    ax.plot(train_sizes, tr_mean, "o-", label="Training score")
    ax.plot(train_sizes, te_mean, "o-", label="CV score")
    ax.set_title(f"{model_name} — {stage} Learning Curve")
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Accuracy")
    ax.grid(True)
    ax.legend(loc="best")

    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_curve(scores_by_n, model_name, metric_name, pdf):
    fig = plt.figure()
    n_vals, sc_vals = zip(*scores_by_n)
    plt.plot(n_vals, sc_vals, marker="o")
    plt.title(f"{model_name} - {metric_name} vs Top N Features (GroupKFold)")
    plt.xlabel("Top N Features")
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_importance(feat_df, model_name, pdf, top_n=20):
    feat_df_sorted = feat_df.sort_values("Importance", ascending=False).head(top_n)
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_df_sorted, x="Importance", y="Feature")
    plt.title(f"{model_name} - Top {top_n} Feature Importances")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)


def _coalesce_col(df: pd.DataFrame, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Missing required column. Tried: {candidates}. "
            f"Found: {sorted(df.columns.tolist())[:120]}..."
        )
    return None

_SPECIES_RE = re.compile(r"^P([EFKR])\.Q([HML])$", re.IGNORECASE)

def derive_species_from_combo(combo: str) -> str:
    """
    Expected mixtures combo format:
      PK.QH, PE.QM, PR.QL, PF.QH, ...
    Returns:
      E/F/K/R
    """
    if combo is None:
        return None
    s = str(combo).strip().upper().replace(" ", "")
    m = _SPECIES_RE.match(s)
    return m.group(1) if m else None

def load_pivot_table_multiclass(data_file: Path, class_mode: str) -> pd.DataFrame:
    df = pd.read_csv(data_file)

    combo_col = _coalesce_col(df, ["Combo", "COMBO"])
    chip_col = _coalesce_col(df, ["ChipNumber", "Chip", "ChipNum", "Chip_Number"])
    channel_col = _coalesce_col(df, ["ChannelNum", "Channel", "ChannelNumber", "Channel_Num"])
    lane_col = _coalesce_col(df, ["Lane", "LaneNum", "LaneNumber", "Lane_Num"])

    df = df.rename(columns={
        combo_col: "Combo",
        chip_col: "ChipNumber",
        channel_col: "ChannelNum",
        lane_col: "Lane",
    }).copy()

    df["Combo"] = df["Combo"].astype(str).str.strip().str.upper()
    df["ChipNumber"] = pd.to_numeric(df["ChipNumber"], errors="coerce")
    df["ChannelNum"] = pd.to_numeric(df["ChannelNum"], errors="coerce")
    df["Lane"] = pd.to_numeric(df["Lane"], errors="coerce")

    df = df.dropna(subset=["Combo", "ChipNumber", "ChannelNum", "Lane"]).copy()
    df["ChipNumber"] = df["ChipNumber"].astype(int)
    df["ChannelNum"] = df["ChannelNum"].astype(int)
    df["Lane"] = df["Lane"].astype(int)

    df["GroupLabel"] = (
        df["Combo"]
        + "_Chip" + df["ChipNumber"].astype(str)
        + "_Chan" + df["ChannelNum"].astype(str)
    )

    if class_mode.lower() == "combo":
        df["Label"] = df["Combo"]

    elif class_mode.lower() == "species":
        df["Label"] = df["Combo"].apply(derive_species_from_combo)

        bad = df["Label"].isna().sum()
        if bad > 0:
            ex = (
                df.loc[df["Label"].isna(), "Combo"]
                .astype(str)
                .drop_duplicates()
                .sort_values()
                .head(50)
                .tolist()
            )
            raise ValueError(
                f"Could not derive species for {bad} rows.\n"
                f"Examples that failed:\n{ex}\n\n"
                f"Expected format like PK.QH / PE.QM / PR.QL / PF.QH"
            )

        bad2 = (~df["Label"].isin(["E", "F", "K", "R"])).sum()
        if bad2 > 0:
            ex2 = df.loc[~df["Label"].isin(["E", "F", "K", "R"]), "Label"].value_counts().to_dict()
            raise ValueError(f"Unexpected species labels found: {ex2}")

    else:
        raise ValueError("CLASS_MODE must be 'combo' or 'species'")

    return df

META_COLS = ["Combo", "Label", "ChipNumber", "ChannelNum", "Lane", "GroupLabel"]

def make_X(sub: pd.DataFrame) -> pd.DataFrame:
    X = (
        sub.drop(columns=[c for c in META_COLS if c in sub.columns], errors="ignore")
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return X


def rank_features_generic(model, X, y, rng_seed):
    """
    Uses model-derived feature importance if possible.
    Falls back to Logistic Regression coefficients.
    """
    try:
        m = clone(model)
        if hasattr(m, "random_state"):
            try:
                m.set_params(random_state=rng_seed)
            except Exception:
                pass
        m.fit(X, y)

        if hasattr(m, "feature_importances_"):
            imp = np.asarray(m.feature_importances_)
        elif hasattr(m, "coef_"):
            coef = np.asarray(m.coef_)
            if coef.ndim == 1:
                imp = np.abs(coef)
            else:
                imp = np.abs(coef).mean(axis=0)
        else:
            raise AttributeError("No feature importance or coefficients.")
    except Exception:
        lr = LogisticRegression(max_iter=2000, solver="lbfgs", multi_class="multinomial", random_state=rng_seed)
        lr.fit(X, y)
        coef = np.asarray(lr.coef_)
        if coef.ndim == 1:
            imp = np.abs(coef)
        else:
            imp = np.abs(coef).mean(axis=0)

    imp = np.ravel(imp)[:X.shape[1]]

    return (
        pd.DataFrame({"Feature": X.columns, "Importance": imp})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )



def compute_metrics(y_true, y_pred, task_type="multiclass"):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "MacroF1": f1_score(y_true, y_pred, average="macro"),
    }



def resumable_topn_scan(
    model_name,
    base_model,
    X,
    y,
    groups,
    sorted_features,
    gk_splits,
    results_dir,
    rng_seed,
    primary_scoring="accuracy",
    task_type="multiclass",
):
    progress_csv = Path(results_dir) / f"TopFeatures_{model_name}_Train.csv"
    rows = []
    tried_n = set()

    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if not df_done.empty and "Top_N" in df_done.columns:
            rows = df_done.to_dict(orient="records")
            tried_n = set(df_done["Top_N"].astype(int).tolist())
            print(f"🔄 Resuming Top-N scan for {model_name}: {len(tried_n)} already done.")

    gkf = GroupKFold(n_splits=gk_splits)
    max_n = min(TOPN_MAX, len(sorted_features), X.shape[1])

    for n in range(1, max_n + 1):
        if n in tried_n:
            continue

        top_feats = sorted_features[:n]
        pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(base_model))])

        if hasattr(pipe.named_steps["est"], "random_state"):
            try:
                pipe.named_steps["est"].set_params(random_state=rng_seed)
            except Exception:
                pass

        preds = cross_val_predict(pipe, X[top_feats], y, cv=gkf, groups=groups, n_jobs=-1)
        metric_dict = compute_metrics(y, preds, task_type=task_type)

        primary_value = metric_dict["Accuracy"] if primary_scoring == "accuracy" else metric_dict["MacroF1"]

        row = {
            "Top_N": int(n),
            "Primary": float(primary_value),
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
        }

        rows.append(row)
        pd.DataFrame(rows).to_csv(progress_csv, index=False)

        print(
            f"[{model_name}] Top {n} -> primary={row['Primary']:.4f} | "
            f"acc={row['Accuracy']:.4f} | macroF1={row['MacroF1']:.4f}"
        )

    return pd.DataFrame(rows)


def resumable_group_grid_search(
    model_name,
    base_estimator,
    param_grid,
    X,
    y,
    groups,
    results_dir,
    scoring="accuracy",
    cv_splits=5,
    random_state=42,
    tag="",
):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    progress_csv = results_dir / f"GridSearch_Resume_{model_name}{tag}.csv"
    best_params_json = results_dir / f"BestParams_{model_name}{tag}.json"

    combos = list(ParameterGrid(param_grid)) if param_grid else [{}]
    gkf = GroupKFold(n_splits=cv_splits)

    tried = set()
    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if "params_json" in df_done.columns:
            tried = set(df_done["params_json"].astype(str).tolist())

    best_score, best_params = -np.inf, None
    if best_params_json.exists():
        try:
            prev = json.load(open(best_params_json))
            best_params = prev.get("best_params")
            best_score = prev.get("best_score", -np.inf)
        except Exception:
            pass

    print(f"🔎 {model_name}: {len(combos)} combos total; {len(tried)} already done.")

    for i, params in enumerate(combos, 1):
        key = json.dumps(params, sort_keys=True)
        if key in tried:
            continue

        est = clone(base_estimator)
        if hasattr(est, "random_state"):
            try:
                est.set_params(random_state=random_state)
            except Exception:
                pass

        try:
            est.set_params(**params)
        except ValueError as e:
            pd.DataFrame([{
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model": model_name,
                "params_json": key,
                "mean_score": np.nan,
                "std_score": np.nan,
                "n_folds": 0,
                "note": f"incompatible: {e}",
            }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())
            continue

        pipe = Pipeline([("scaler", StandardScaler()), ("est", est)])
        scores = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring=scoring, n_jobs=-1)
        mean_score, std_score = float(np.mean(scores)), float(np.std(scores))

        pd.DataFrame([{
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "model": model_name,
            "params_json": key,
            "mean_score": mean_score,
            "std_score": std_score,
            "n_folds": len(scores),
        }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())

        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            json.dump({"best_params": best_params, "best_score": best_score}, open(best_params_json, "w"))

        print(
            f"[{model_name}] {i}/{len(combos)} params={params} -> "
            f"{scoring}={mean_score:.4f} (±{std_score:.4f}) | best={best_score:.4f}"
        )

    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("est", clone(base_estimator).set_params(**(best_params or {})))
    ])

    return best_pipe, best_params, best_score


def evaluate_stage_loocv(
    model,
    X,
    y,
    groups,
    model_name,
    stage,
    cmap,
    labels,
    pdf,
    results_dir,
    seed,
    data_file_name,
    task_type="multiclass",
):
    logo = LeaveOneGroupOut()
    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(model))])

    if hasattr(pipe.named_steps["est"], "random_state"):
        try:
            pipe.named_steps["est"].set_params(random_state=seed)
        except Exception:
            pass

    preds = cross_val_predict(pipe, X, y, cv=logo, groups=groups, n_jobs=-1)
    metric_dict = compute_metrics(y, preds, task_type=task_type)

    report_str = classification_report(y, preds, target_names=labels)

    print(
        f"\n{model_name} - {stage} LOOCV "
        f"| Acc={metric_dict['Accuracy']:.3f} "
        f"| MacroF1={metric_dict['MacroF1']:.3f}"
    )

    add_text_page(
        pdf,
        f"{model_name} — {stage} LOOCV (seed {seed})",
        report_str
    )

    fig_cm = plt.figure()
    sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d",
                cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(f"{model_name} - {stage} Confusion Matrix (LOOCV, seed {seed})")
    plt.tight_layout()
    pdf.savefig(fig_cm)
    plt.show()
    plt.close(fig_cm)

    try:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="predict_proba", n_jobs=-1)
    except Exception:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="decision_function", n_jobs=-1)
        y_scores = np.asarray(y_scores)
        if y_scores.ndim == 1:
            y_scores = y_scores.reshape(-1, 1)

    class_ids = np.arange(len(labels))
    Y = label_binarize(y, classes=class_ids)
    S = np.asarray(y_scores)

    if S.ndim == 1:
        S = S.reshape(-1, 1)

    if S.max() > 1.0 or S.min() < 0.0:
        S = (S - S.min(axis=0)) / (S.max(axis=0) - S.min(axis=0) + 1e-9)

    fig_roc, ax_roc = plt.subplots()
    fig_pr, ax_pr = plt.subplots()

    for i, label in enumerate(labels):
        fpr, tpr, _ = roc_curve(Y[:, i], S[:, i])
        prec, rec, _ = precision_recall_curve(Y[:, i], S[:, i])
        ax_roc.plot(fpr, tpr, label=f"{label} (AUC={auc(fpr, tpr):.2f})")
        ax_pr.plot(rec, prec, label=f"{label} (AUC={auc(rec, prec):.2f})")

    ax_roc.plot([0, 1], [0, 1], "k--")
    ax_roc.set_title(f"{model_name} - {stage} ROC (LOOCV, seed {seed})")
    ax_pr.set_title(f"{model_name} - {stage} PR (LOOCV, seed {seed})")
    ax_roc.set_xlabel("FPR")
    ax_roc.set_ylabel("TPR")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_roc.legend()
    ax_pr.legend()

    plt.tight_layout()
    pdf.savefig(fig_roc)
    pdf.savefig(fig_pr)
    plt.show()
    plt.close(fig_roc)
    plt.close(fig_pr)

    save_metrics_row_csv(
        {
            "Model": model_name,
            "Stage": stage,
            "Seed": seed,
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
            "DataFile": data_file_name,
        },
        Path(results_dir) / "All_Metrics_LOOCV.csv"
    )

    return metric_dict, preds


def evaluate_model_end_to_end(
    model_name,
    base_model,
    X,
    y,
    groups,
    labels,
    cmap,
    gk_splits,
    rng_seed,
    results_dir_seed,
    data_file_name,
    primary_scoring="accuracy",
    task_type="multiclass",
):
    pdf_path = Path(results_dir_seed) / f"{model_name}_Results_Seed{rng_seed}.pdf"
    summary = {
        "Model": model_name,
        "Seed": rng_seed,
        "Task": TASK_NAME,
        "TaskType": task_type,
        "PrimaryScoring": primary_scoring,
        "ClassMode": CLASS_MODE,
    }

    gkf_for_curves = GroupKFold(n_splits=gk_splits)

    with PdfPages(pdf_path) as pdf:


        baseline_metrics, _ = evaluate_stage_loocv(
            base_model, X, y, groups,
            model_name, "Baseline", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Baseline_LOOCV_Accuracy"] = baseline_metrics["Accuracy"]
        summary["Baseline_LOOCV_MacroF1"] = baseline_metrics["MacroF1"]

        plot_learning_curve_cv(
            base_model, X, y, groups, model_name, "Baseline", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )


        feat_df = rank_features_generic(base_model, X, y, rng_seed)
        sorted_names = feat_df["Feature"].tolist()[:min(TOPN_MAX, X.shape[1])]

        df_topn = resumable_topn_scan(
            model_name=model_name,
            base_model=base_model,
            X=X,
            y=y,
            groups=groups,
            sorted_features=sorted_names,
            gk_splits=gk_splits,
            results_dir=results_dir_seed,
            rng_seed=rng_seed,
            primary_scoring=primary_scoring,
            task_type=task_type,
        )

        best_row = df_topn.sort_values("Primary", ascending=False).iloc[0]
        best_n = int(best_row["Top_N"])
        top_features = sorted_names[:best_n]
        summary["TopN"] = best_n

        plot_feature_curve(
            list(zip(df_topn["Top_N"].tolist(), df_topn["Primary"].tolist())),
            model_name,
            f"Primary({primary_scoring})",
            pdf
        )
        plot_feature_importance(feat_df, model_name, pdf)

        pd.DataFrame({"Feature": top_features}).to_csv(
            Path(results_dir_seed) / f"TopFeatures_{model_name}_Selected.csv",
            index=False
        )

        top_metrics, _ = evaluate_stage_loocv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["TopFeatures_LOOCV_Accuracy"] = top_metrics["Accuracy"]
        summary["TopFeatures_LOOCV_MacroF1"] = top_metrics["MacroF1"]

        plot_learning_curve_cv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        _, best_params, _ = resumable_group_grid_search(
            model_name=model_name,
            base_estimator=base_model,
            param_grid=GRID.get(model_name, {}),
            X=X[top_features],
            y=y,
            groups=groups,
            results_dir=results_dir_seed,
            scoring=primary_scoring,
            cv_splits=gk_splits,
            random_state=rng_seed,
        )

        tuned_best = clone(base_model).set_params(**(best_params or {}))
        if hasattr(tuned_best, "random_state"):
            try:
                tuned_best.set_params(random_state=rng_seed)
            except Exception:
                pass

        tuned_metrics, _ = evaluate_stage_loocv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Tuned_LOOCV_Accuracy"] = tuned_metrics["Accuracy"]
        summary["Tuned_LOOCV_MacroF1"] = tuned_metrics["MacroF1"]
        summary["Best_Params"] = best_params

        plot_learning_curve_cv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )


        final_pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(tuned_best))])
        final_pipe.fit(X[top_features], y)

        artifact = {
            "pipeline": final_pipe,
            "top_features": top_features,
            "labels": labels,
            "best_params": best_params,
            "data_file": data_file_name,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "seed": rng_seed,
            "task": TASK_NAME,
            "task_type": task_type,
            "primary_scoring": primary_scoring,
            "class_mode": CLASS_MODE,
        }
        joblib.dump(artifact, Path(results_dir_seed) / f"{model_name}_Final_Fitted.joblib")

    return summary


def print_group_summary(X, y, groups, labels=None, dataset_name="Dataset"):
    import pandas as pd
    import numpy as np

    groups_series = pd.Series(groups).astype(str)
    y_array = np.asarray(y)

    print("\n" + "=" * 70)
    print(f"{dataset_name} SUMMARY")
    print("=" * 70)
    print(f"Rows: {len(y_array):,}")
    print(f"Features: {X.shape[1]:,}")
    print(f"Unique groups: {groups_series.nunique():,}")

    group_sizes = groups_series.value_counts()
    print(f"Min rows per group: {group_sizes.min():,}")
    print(f"Max rows per group: {group_sizes.max():,}")
    print(f"Mean rows per group: {group_sizes.mean():.2f}")

    if labels is not None:
        print(f"Classes ({len(labels)}): {labels}")

    print("\nFirst 10 groups:")
    print(group_sizes.head(10).to_string())

    print("=" * 70 + "\n")


# MAIN

if __name__ == "__main__":
    df_all = load_pivot_table_multiclass(DATA_FILE, CLASS_MODE)

    print(f"\n DATA_FILE={DATA_FILE}")
    print(f" Rows: {len(df_all):,}")
    print(f" Groups: {df_all['GroupLabel'].nunique():,}")

    X = make_X(df_all)

    if X.shape[1] == 0:
        raise RuntimeError("No numeric feature columns found. Check pivot table export.")

    le = LabelEncoder()
    labels_all = sorted(df_all["Label"].unique())
    le.fit(labels_all)
    y = le.transform(df_all["Label"])
    labels = list(le.classes_)
    groups = df_all["GroupLabel"].astype(str).values

    print_group_summary(
    X=X,
    y=y,
    groups=groups,
    labels=labels,
    dataset_name=f"Mixtures Multiclass ({CLASS_MODE})"
)

    print(f"  ClassMode = {CLASS_MODE}")
    print(f"  Classes ({len(labels)}): {labels}")
    print(f" Features available = {X.shape[1]}")
    print(f" Primary scoring = {PRIMARY_SCORING}")
    print(f" XGBoost available = {HAS_XGB}")

    all_summaries = []

    for seed in SEED_LIST:
        print(f"\n SEED {seed} ")
        results_dir_seed = RESULTS_DIR / f"Seed_{seed}"
        results_dir_seed.mkdir(parents=True, exist_ok=True)

        MODELS = make_models_for_seed(seed)

        per_seed_summaries = []

        for model_name, model in MODELS.items():
            print(f"\n🚀 Seed {seed} — Running {model_name}")

            cmap = MODEL_CMAPS.get(model_name, plt.get_cmap("Blues"))

            summary = evaluate_model_end_to_end(
                model_name=model_name,
                base_model=model,
                X=X,
                y=y,
                groups=groups,
                labels=labels,
                cmap=cmap,
                gk_splits=GK_N_SPLITS,
                rng_seed=seed,
                results_dir_seed=results_dir_seed,
                data_file_name=Path(DATA_FILE).name,
                primary_scoring=PRIMARY_SCORING,
                task_type=TASK_TYPE,
            )
            per_seed_summaries.append(summary)
            all_summaries.append(summary)

        per_seed_df = pd.DataFrame(per_seed_summaries)
        col_order = [
            "Model",
            "Seed",
            "Task",
            "TaskType",
            "PrimaryScoring",
            "ClassMode",
            "Baseline_LOOCV_Accuracy",
            "Baseline_LOOCV_MacroF1",
            "TopN",
            "TopFeatures_LOOCV_Accuracy",
            "TopFeatures_LOOCV_MacroF1",
            "Tuned_LOOCV_Accuracy",
            "Tuned_LOOCV_MacroF1",
            "Best_Params",
        ]
        cols_present = [c for c in col_order if c in per_seed_df.columns]
        per_seed_df = per_seed_df.loc[:, cols_present + [c for c in per_seed_df.columns if c not in cols_present]]

        per_seed_path = results_dir_seed / "Model_Comparison_Summary_Multiclass_Train.csv"
        per_seed_df.to_csv(per_seed_path, index=False)
        print(f"📝 Seed {seed} summary saved → {per_seed_path}")

    summary_df = pd.DataFrame(all_summaries)
    col_order = [
        "Model",
        "Seed",
        "Task",
        "TaskType",
        "PrimaryScoring",
        "ClassMode",
        "Baseline_LOOCV_Accuracy",
        "Baseline_LOOCV_MacroF1",
        "TopN",
        "TopFeatures_LOOCV_Accuracy",
        "TopFeatures_LOOCV_MacroF1",
        "Tuned_LOOCV_Accuracy",
        "Tuned_LOOCV_MacroF1",
        "Best_Params",
    ]
    cols_present = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df.loc[:, cols_present + [c for c in summary_df.columns if c not in cols_present]]

    summary_path = RESULTS_DIR / "Model_Comparison_Summary_Multiclass_AllSeeds.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\n DONE.")

##**Bacteria Mixtures Binary**

In [ ]:
'''#!/usr/bin/env python
# coding: utf-8
 BACTERIA MIXTURES (Binary) — Unified Group-aware LOOCV + Top-N + Tuning

 TASK
 - Binary classification:
     Healthy = K
     Dysbiotic = R / E / F

 STANDARDIZED PIPELINE
 1) Baseline LOOCV on all features
 2) Top-N scan using GroupKFold over N = 1..360 (no skipping)
 3) LOOCV on best Top-N feature subset
 4) GroupKFold grid search on Top-N features
 5) LOOCV on tuned model
 6) Fit final tuned model on all data and save artifact

 PRIMARY MODEL-SELECTION METRIC
 - Macro F1 (f1_macro)

 - Accuracy
 - Macro F1
 - Positive-class F1 (Dysbiotic = 1)

 GROUPING
- LeaveOneGroupOut / GroupKFold groups are:
#     Combo + ChipNumber + ChannelNum
'''

import os
import json
import time
import warnings
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    ParameterGrid,
    cross_val_predict,
    cross_val_score,
    learning_curve,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")



FORCE_RERUN = False
GK_N_SPLITS = 5
SEED_LIST = list(range(1, 3))
TOPN_MAX = 360

TASK_NAME = "Bacteria Mixtures Binary: K vs Dysbiotic(REF)"
TASK_TYPE = "binary"
PRIMARY_SCORING = "f1_macro"   # primary metric for Top-N + tuning

BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/(Redo) Bacterial Mixtures in PBS")
DATA_FILE = BASE_FOLDER / "ML_Tables" / "ML_Aligned_Complete_Pivot_Table_Mixtures_withoutNaCl_aic_45s_1vps_031922.csv"
RESULTS_DIR = BASE_FOLDER / "ML_Results" / "Mixtures_GroupLOOCV_Binary_K_vs_Dysbiotic_MacroF1_Unified_360TopN_03192026"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CMAPS = {
    "Random Forest": plt.get_cmap("Purples"),
    "SVM": plt.get_cmap("Greens"),
    "Logistic Regression": plt.get_cmap("Oranges"),
    "LDA": plt.get_cmap("YlOrBr"),
    "XGBoost": plt.get_cmap("Blues"),
}


GRID = {
    "Random Forest": {
        "n_estimators": [200, 400, 800],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced"],
    },

    "SVM": [
        {
            "kernel": ["rbf"],
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", 0.01, 0.001],
            "class_weight": [None, "balanced"],
        },
        {
            "kernel": ["linear"],
            "C": [0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
        },
    ],

    "Logistic Regression": [
        {
            "solver": ["lbfgs"],
            "penalty": ["l2"],
            "C": [0.01, 0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
        },
        {
            "solver": ["saga"],
            "penalty": ["l1", "l2", "elasticnet"],
            "C": [0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0.0, 0.5, 1.0],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
        },
    ],

    "LDA": [
        {"solver": ["svd"]},
        {"solver": ["lsqr", "eigen"], "shrinkage": ["auto", 0.1, 0.5]},
    ],
}

if HAS_XGB:
    GRID["XGBoost"] = {
        "n_estimators": [100, 200, 400],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }



def make_models_for_seed(seed: int):
    models = {
        "Random Forest": RandomForestClassifier(random_state=seed),
        "SVM": SVC(kernel="rbf", probability=True, random_state=seed),
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=seed
        ),
        "LDA": LinearDiscriminantAnalysis(),
    }
    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            eval_metric="mlogloss",
            random_state=seed
        )
    return models



def add_text_page(pdf, title, text, fontsize=9):
    fig = plt.figure(figsize=(8.5, 11))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    fig.text(0.5, 0.97, title, ha="center", va="top", fontsize=11, weight="bold")
    fig.text(0.02, 0.93, text, ha="left", va="top", fontsize=fontsize, family="monospace")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

def save_metrics_row_csv(row_dict, out_csv):
    out_csv = Path(out_csv)
    df = pd.DataFrame([row_dict])
    df.to_csv(out_csv, mode="a", index=False, header=not out_csv.exists())

def plot_learning_curve_cv(estimator, X, y, groups, model_name, stage, pdf,
                           cv=None, n_points=10, rng_seed=42):
    if cv is None:
        cv = GroupKFold(n_splits=GK_N_SPLITS)

    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(estimator))])

    fig, ax = plt.subplots(figsize=(7, 5))
    train_sizes, train_scores, test_scores = learning_curve(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, n_points),
        shuffle=True,
        random_state=rng_seed,
    )
    tr_mean, tr_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
    te_mean, te_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)

    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)
    ax.fill_between(train_sizes, te_mean - te_std, te_mean + te_std, alpha=0.1)
    ax.plot(train_sizes, tr_mean, "o-", label="Training score")
    ax.plot(train_sizes, te_mean, "o-", label="CV score")
    ax.set_title(f"{model_name} — {stage} Learning Curve")
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Accuracy")
    ax.grid(True)
    ax.legend(loc="best")

    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_curve(scores_by_n, model_name, metric_name, pdf):
    fig = plt.figure()
    n_vals, sc_vals = zip(*scores_by_n)
    plt.plot(n_vals, sc_vals, marker="o")
    plt.title(f"{model_name} - {metric_name} vs Top N Features (GroupKFold)")
    plt.xlabel("Top N Features")
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_importance(feat_df, model_name, pdf, top_n=20):
    feat_df_sorted = feat_df.sort_values("Importance", ascending=False).head(top_n)
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_df_sorted, x="Importance", y="Feature")
    plt.title(f"{model_name} - Top {top_n} Feature Importances")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)



def _coalesce_col(df: pd.DataFrame, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Missing required column. Tried: {candidates}. "
            f"Found: {sorted(df.columns.tolist())[:120]}..."
        )
    return None

_SPECIES_RE = re.compile(r"^P([EFKR])\.Q([HML])$", re.IGNORECASE)

def derive_species_from_combo(combo: str):
    if combo is None:
        return None
    s = str(combo).strip().upper().replace(" ", "")
    m = _SPECIES_RE.match(s)
    return m.group(1) if m else None

def derive_binary_label_from_species(spec: str):
    """
    Healthy = K -> 0
    Dysbiotic = R/E/F -> 1
    """
    if spec is None:
        return None
    spec = str(spec).strip().upper()
    if spec == "K":
        return 0
    if spec in ("R", "E", "F"):
        return 1
    return None

def load_pivot_table_binary(data_file: Path) -> pd.DataFrame:
    df = pd.read_csv(data_file)

    combo_col = _coalesce_col(df, ["Combo", "COMBO"])
    chip_col = _coalesce_col(df, ["ChipNumber", "Chip", "ChipNum", "Chip_Number"])
    channel_col = _coalesce_col(df, ["ChannelNum", "Channel", "ChannelNumber", "Channel_Num"])
    lane_col = _coalesce_col(df, ["Lane", "LaneNum", "LaneNumber", "Lane_Num"])

    df = df.rename(columns={
        combo_col: "Combo",
        chip_col: "ChipNumber",
        channel_col: "ChannelNum",
        lane_col: "Lane",
    }).copy()

    df["Combo"] = df["Combo"].astype(str).str.strip().str.upper()
    df["ChipNumber"] = pd.to_numeric(df["ChipNumber"], errors="coerce")
    df["ChannelNum"] = pd.to_numeric(df["ChannelNum"], errors="coerce")
    df["Lane"] = pd.to_numeric(df["Lane"], errors="coerce")

    df = df.dropna(subset=["Combo", "ChipNumber", "ChannelNum", "Lane"]).copy()
    df["ChipNumber"] = df["ChipNumber"].astype(int)
    df["ChannelNum"] = df["ChannelNum"].astype(int)
    df["Lane"] = df["Lane"].astype(int)

    df["Species"] = df["Combo"].apply(derive_species_from_combo)
    bad = df["Species"].isna().sum()
    if bad > 0:
        ex = (
            df.loc[df["Species"].isna(), "Combo"]
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .head(50)
            .tolist()
        )
        raise ValueError(
            f"Could not derive species for {bad} rows.\n"
            f"Examples that failed:\n{ex}\n\n"
            f"Expected format like PK.QH / PE.QM / PR.QL / PF.QH"
        )

    df["y"] = df["Species"].apply(derive_binary_label_from_species)
    bad2 = df["y"].isna().sum()
    if bad2 > 0:
        ex2 = (
            df.loc[df["y"].isna(), ["Combo", "Species"]]
            .drop_duplicates()
            .head(20)
            .to_dict(orient="records")
        )
        raise ValueError(f"Unexpected labels while building binary target. Examples: {ex2}")

    df["GroupLabel"] = (
        df["Combo"]
        + "_Chip" + df["ChipNumber"].astype(str)
        + "_Chan" + df["ChannelNum"].astype(str)
    )

    return df

META_COLS = ["Combo", "Species", "y", "ChipNumber", "ChannelNum", "Lane", "GroupLabel"]

def make_X(sub: pd.DataFrame) -> pd.DataFrame:
    X = (
        sub.drop(columns=[c for c in META_COLS if c in sub.columns], errors="ignore")
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return X


def rank_features_generic(model, X, y, rng_seed):
    """
    Uses model-derived feature importance if possible.
    Falls back to Logistic Regression coefficients.
    """
    try:
        m = clone(model)
        if hasattr(m, "random_state"):
            try:
                m.set_params(random_state=rng_seed)
            except Exception:
                pass
        m.fit(X, y)

        if hasattr(m, "feature_importances_"):
            imp = np.asarray(m.feature_importances_)
        elif hasattr(m, "coef_"):
            coef = np.asarray(m.coef_)
            if coef.ndim == 1:
                imp = np.abs(coef)
            else:
                imp = np.abs(coef).mean(axis=0)
        else:
            raise AttributeError("No feature importance or coefficients.")
    except Exception:
        lr = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=rng_seed)
        lr.fit(X, y)
        coef = np.asarray(lr.coef_)
        if coef.ndim == 1:
            imp = np.abs(coef)
        else:
            imp = np.abs(coef).mean(axis=0)

    imp = np.ravel(imp)[:X.shape[1]]

    return (
        pd.DataFrame({"Feature": X.columns, "Importance": imp})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )


def compute_metrics(y_true, y_pred, task_type="binary"):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "MacroF1": f1_score(y_true, y_pred, average="macro"),
    }
    if task_type == "binary":
        metrics["PositiveClassF1"] = f1_score(y_true, y_pred, pos_label=1)
    return metrics



def resumable_topn_scan(
    model_name,
    base_model,
    X,
    y,
    groups,
    sorted_features,
    gk_splits,
    results_dir,
    rng_seed,
    primary_scoring="accuracy",
    task_type="binary",
):
    progress_csv = Path(results_dir) / f"TopFeatures_{model_name}_Train.csv"
    rows = []
    tried_n = set()

    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if not df_done.empty and "Top_N" in df_done.columns:
            rows = df_done.to_dict(orient="records")
            tried_n = set(df_done["Top_N"].astype(int).tolist())
            print(f"🔄 Resuming Top-N scan for {model_name}: {len(tried_n)} already done.")

    gkf = GroupKFold(n_splits=gk_splits)
    max_n = min(TOPN_MAX, len(sorted_features), X.shape[1])

    for n in range(1, max_n + 1):
        if n in tried_n:
            continue

        top_feats = sorted_features[:n]
        pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(base_model))])

        if hasattr(pipe.named_steps["est"], "random_state"):
            try:
                pipe.named_steps["est"].set_params(random_state=rng_seed)
            except Exception:
                pass

        preds = cross_val_predict(pipe, X[top_feats], y, cv=gkf, groups=groups, n_jobs=-1)
        metric_dict = compute_metrics(y, preds, task_type=task_type)

        primary_value = metric_dict["MacroF1"] if primary_scoring == "f1_macro" else metric_dict["Accuracy"]

        row = {
            "Top_N": int(n),
            "Primary": float(primary_value),
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
        }
        if task_type == "binary":
            row["PositiveClassF1"] = float(metric_dict["PositiveClassF1"])

        rows.append(row)
        pd.DataFrame(rows).to_csv(progress_csv, index=False)

        print(
            f"[{model_name}] Top {n} -> primary={row['Primary']:.4f} | "
            f"acc={row['Accuracy']:.4f} | macroF1={row['MacroF1']:.4f} | "
            f"posF1={row['PositiveClassF1']:.4f}"
        )

    return pd.DataFrame(rows)

def resumable_group_grid_search(
    model_name,
    base_estimator,
    param_grid,
    X,
    y,
    groups,
    results_dir,
    scoring="accuracy",
    cv_splits=5,
    random_state=42,
    tag="",
):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    progress_csv = results_dir / f"GridSearch_Resume_{model_name}{tag}.csv"
    best_params_json = results_dir / f"BestParams_{model_name}{tag}.json"

    combos = list(ParameterGrid(param_grid)) if param_grid else [{}]
    gkf = GroupKFold(n_splits=cv_splits)

    tried = set()
    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if "params_json" in df_done.columns:
            tried = set(df_done["params_json"].astype(str).tolist())

    best_score, best_params = -np.inf, None
    if best_params_json.exists():
        try:
            prev = json.load(open(best_params_json))
            best_params = prev.get("best_params")
            best_score = prev.get("best_score", -np.inf)
        except Exception:
            pass

    print(f"🔎 {model_name}: {len(combos)} combos total; {len(tried)} already done.")

    for i, params in enumerate(combos, 1):
        key = json.dumps(params, sort_keys=True)
        if key in tried:
            continue

        est = clone(base_estimator)
        if hasattr(est, "random_state"):
            try:
                est.set_params(random_state=random_state)
            except Exception:
                pass

        try:
            est.set_params(**params)
        except ValueError as e:
            pd.DataFrame([{
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model": model_name,
                "params_json": key,
                "mean_score": np.nan,
                "std_score": np.nan,
                "n_folds": 0,
                "note": f"incompatible: {e}",
            }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())
            continue

        pipe = Pipeline([("scaler", StandardScaler()), ("est", est)])
        scores = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring=scoring, n_jobs=-1)
        mean_score, std_score = float(np.mean(scores)), float(np.std(scores))

        pd.DataFrame([{
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "model": model_name,
            "params_json": key,
            "mean_score": mean_score,
            "std_score": std_score,
            "n_folds": len(scores),
        }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())

        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            json.dump({"best_params": best_params, "best_score": best_score}, open(best_params_json, "w"))

        print(
            f"[{model_name}] {i}/{len(combos)} params={params} -> "
            f"{scoring}={mean_score:.4f} (±{std_score:.4f}) | best={best_score:.4f}"
        )

    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("est", clone(base_estimator).set_params(**(best_params or {})))
    ])

    return best_pipe, best_params, best_score


def evaluate_stage_loocv(
    model,
    X,
    y,
    groups,
    model_name,
    stage,
    cmap,
    labels,
    pdf,
    results_dir,
    seed,
    data_file_name,
    task_type="binary",
):
    logo = LeaveOneGroupOut()
    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(model))])

    if hasattr(pipe.named_steps["est"], "random_state"):
        try:
            pipe.named_steps["est"].set_params(random_state=seed)
        except Exception:
            pass

    preds = cross_val_predict(pipe, X, y, cv=logo, groups=groups, n_jobs=-1)
    metric_dict = compute_metrics(y, preds, task_type=task_type)

    report_str = classification_report(y, preds, target_names=labels)

    print(
        f"\n{model_name} - {stage} LOOCV "
        f"| Acc={metric_dict['Accuracy']:.3f} "
        f"| MacroF1={metric_dict['MacroF1']:.3f} "
        f"| PosF1={metric_dict['PositiveClassF1']:.3f}"
    )

    add_text_page(
        pdf,
        f"{model_name} — {stage} LOOCV (seed {seed})",
        report_str
    )

    fig_cm = plt.figure()
    sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d",
                cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(f"{model_name} - {stage} Confusion Matrix (LOOCV, seed {seed})")
    plt.tight_layout()
    pdf.savefig(fig_cm)
    plt.show()
    plt.close(fig_cm)

    try:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="predict_proba", n_jobs=-1)[:, 1]
    except Exception:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="decision_function", n_jobs=-1)
        y_scores = np.asarray(y_scores).ravel()

    fpr, tpr, _ = roc_curve(y, y_scores)
    prec, rec, _ = precision_recall_curve(y, y_scores)

    fig_roc, ax_roc = plt.subplots()
    ax_roc.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.2f}")
    ax_roc.plot([0, 1], [0, 1], "k--")
    ax_roc.set_title(f"{model_name} - {stage} ROC (LOOCV, seed {seed})")
    ax_roc.set_xlabel("FPR")
    ax_roc.set_ylabel("TPR")
    ax_roc.legend()
    plt.tight_layout()
    pdf.savefig(fig_roc)
    plt.show()
    plt.close(fig_roc)

    fig_pr, ax_pr = plt.subplots()
    ax_pr.plot(rec, prec, label=f"AUC={auc(rec, prec):.2f}")
    ax_pr.set_title(f"{model_name} - {stage} PR (LOOCV, seed {seed})")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend()
    plt.tight_layout()
    pdf.savefig(fig_pr)
    plt.show()
    plt.close(fig_pr)

    save_metrics_row_csv(
        {
            "Model": model_name,
            "Stage": stage,
            "Seed": seed,
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
            "PositiveClassF1": float(metric_dict["PositiveClassF1"]),
            "DataFile": data_file_name,
        },
        Path(results_dir) / "All_Metrics_LOOCV.csv"
    )

    return metric_dict, preds



def evaluate_model_end_to_end(
    model_name,
    base_model,
    X,
    y,
    groups,
    labels,
    cmap,
    gk_splits,
    rng_seed,
    results_dir_seed,
    data_file_name,
    primary_scoring="accuracy",
    task_type="binary",
):
    pdf_path = Path(results_dir_seed) / f"{model_name}_Results_Seed{rng_seed}.pdf"
    summary = {
        "Model": model_name,
        "Seed": rng_seed,
        "Task": TASK_NAME,
        "PrimaryScoring": primary_scoring,
        "TaskType": task_type,
    }

    gkf_for_curves = GroupKFold(n_splits=gk_splits)

    with PdfPages(pdf_path) as pdf:

        baseline_metrics, _ = evaluate_stage_loocv(
            base_model, X, y, groups,
            model_name, "Baseline", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Baseline_LOOCV_Accuracy"] = baseline_metrics["Accuracy"]
        summary["Baseline_LOOCV_MacroF1"] = baseline_metrics["MacroF1"]
        summary["Baseline_LOOCV_PositiveClassF1"] = baseline_metrics["PositiveClassF1"]

        plot_learning_curve_cv(
            base_model, X, y, groups, model_name, "Baseline", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        feat_df = rank_features_generic(base_model, X, y, rng_seed)
        sorted_names = feat_df["Feature"].tolist()[:min(TOPN_MAX, X.shape[1])]

        df_topn = resumable_topn_scan(
            model_name=model_name,
            base_model=base_model,
            X=X,
            y=y,
            groups=groups,
            sorted_features=sorted_names,
            gk_splits=gk_splits,
            results_dir=results_dir_seed,
            rng_seed=rng_seed,
            primary_scoring=primary_scoring,
            task_type=task_type,
        )

        best_row = df_topn.sort_values("Primary", ascending=False).iloc[0]
        best_n = int(best_row["Top_N"])
        top_features = sorted_names[:best_n]
        summary["TopN"] = best_n

        plot_feature_curve(
            list(zip(df_topn["Top_N"].tolist(), df_topn["Primary"].tolist())),
            model_name,
            f"Primary({primary_scoring})",
            pdf
        )
        plot_feature_importance(feat_df, model_name, pdf)

        pd.DataFrame({"Feature": top_features}).to_csv(
            Path(results_dir_seed) / f"TopFeatures_{model_name}_Selected.csv",
            index=False
        )

        top_metrics, _ = evaluate_stage_loocv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["TopFeatures_LOOCV_Accuracy"] = top_metrics["Accuracy"]
        summary["TopFeatures_LOOCV_MacroF1"] = top_metrics["MacroF1"]
        summary["TopFeatures_LOOCV_PositiveClassF1"] = top_metrics["PositiveClassF1"]

        plot_learning_curve_cv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        _, best_params, _ = resumable_group_grid_search(
            model_name=model_name,
            base_estimator=base_model,
            param_grid=GRID.get(model_name, {}),
            X=X[top_features],
            y=y,
            groups=groups,
            results_dir=results_dir_seed,
            scoring=primary_scoring,
            cv_splits=gk_splits,
            random_state=rng_seed,
        )

        tuned_best = clone(base_model).set_params(**(best_params or {}))
        if hasattr(tuned_best, "random_state"):
            try:
                tuned_best.set_params(random_state=rng_seed)
            except Exception:
                pass

        tuned_metrics, _ = evaluate_stage_loocv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Tuned_LOOCV_Accuracy"] = tuned_metrics["Accuracy"]
        summary["Tuned_LOOCV_MacroF1"] = tuned_metrics["MacroF1"]
        summary["Tuned_LOOCV_PositiveClassF1"] = tuned_metrics["PositiveClassF1"]
        summary["Best_Params"] = best_params

        plot_learning_curve_cv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )


        final_pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(tuned_best))])
        final_pipe.fit(X[top_features], y)

        artifact = {
            "pipeline": final_pipe,
            "top_features": top_features,
            "labels": labels,
            "best_params": best_params,
            "data_file": data_file_name,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "seed": rng_seed,
            "task": TASK_NAME,
            "primary_scoring": primary_scoring,
            "task_type": task_type,
        }
        joblib.dump(artifact, Path(results_dir_seed) / f"{model_name}_Final_Fitted.joblib")

    return summary

# MAIN

if __name__ == "__main__":
    df_all = load_pivot_table_binary(DATA_FILE)

    print(f"\n📄 DATA_FILE={DATA_FILE}")
    print(f" Rows: {len(df_all):,}")
    print(f" Groups: {df_all['GroupLabel'].nunique():,}")

    X = make_X(df_all)
    y = df_all["y"].astype(int).values
    groups = df_all["GroupLabel"].astype(str).values

    if X.shape[1] == 0:
        raise RuntimeError("No numeric feature columns found. Check pivot table export.")

    n0 = int(np.sum(y == 0))
    n1 = int(np.sum(y == 1))
    print(f"  Healthy(K)=0 count={n0} | Dysbiotic(REF)=1 count={n1}")
    print(f" Features available = {X.shape[1]}")
    print(f" Primary scoring = {PRIMARY_SCORING}")
    print(f" XGBoost available = {HAS_XGB}")

    labels = ["Healthy (K)", "Dysbiotic (REF)"]

    all_summaries = []

    for seed in SEED_LIST:
        print(f"\nSEED {seed})
        results_dir_seed = RESULTS_DIR / f"Seed_{seed}"
        results_dir_seed.mkdir(parents=True, exist_ok=True)

        MODELS = make_models_for_seed(seed)

        per_seed_summaries = []

        for model_name, model in MODELS.items():
            print(f"\n🚀 Seed {seed} — Running {model_name}")

            cmap = MODEL_CMAPS.get(model_name, plt.get_cmap("Blues"))

            summary = evaluate_model_end_to_end(
                model_name=model_name,
                base_model=model,
                X=X,
                y=y,
                groups=groups,
                labels=labels,
                cmap=cmap,
                gk_splits=GK_N_SPLITS,
                rng_seed=seed,
                results_dir_seed=results_dir_seed,
                data_file_name=Path(DATA_FILE).name,
                primary_scoring=PRIMARY_SCORING,
                task_type=TASK_TYPE,
            )
            per_seed_summaries.append(summary)
            all_summaries.append(summary)

        per_seed_df = pd.DataFrame(per_seed_summaries)
        col_order = [
            "Model",
            "Seed",
            "Task",
            "TaskType",
            "PrimaryScoring",
            "Baseline_LOOCV_Accuracy",
            "Baseline_LOOCV_MacroF1",
            "Baseline_LOOCV_PositiveClassF1",
            "TopN",
            "TopFeatures_LOOCV_Accuracy",
            "TopFeatures_LOOCV_MacroF1",
            "TopFeatures_LOOCV_PositiveClassF1",
            "Tuned_LOOCV_Accuracy",
            "Tuned_LOOCV_MacroF1",
            "Tuned_LOOCV_PositiveClassF1",
            "Best_Params",
        ]
        cols_present = [c for c in col_order if c in per_seed_df.columns]
        per_seed_df = per_seed_df.loc[:, cols_present + [c for c in per_seed_df.columns if c not in cols_present]]

        per_seed_path = results_dir_seed / "Model_Comparison_Summary_Binary_Train.csv"
        per_seed_df.to_csv(per_seed_path, index=False)
        print(f" Seed {seed} summary saved → {per_seed_path}")

    summary_df = pd.DataFrame(all_summaries)
    col_order = [
        "Model",
        "Seed",
        "Task",
        "TaskType",
        "PrimaryScoring",
        "Baseline_LOOCV_Accuracy",
        "Baseline_LOOCV_MacroF1",
        "Baseline_LOOCV_PositiveClassF1",
        "TopN",
        "TopFeatures_LOOCV_Accuracy",
        "TopFeatures_LOOCV_MacroF1",
        "TopFeatures_LOOCV_PositiveClassF1",
        "Tuned_LOOCV_Accuracy",
        "Tuned_LOOCV_MacroF1",
        "Tuned_LOOCV_PositiveClassF1",
        "Best_Params",
    ]
    cols_present = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df.loc[:, cols_present + [c for c in summary_df.columns if c not in cols_present]]

    summary_path = RESULTS_DIR / "Model_Comparison_Summary_Binary_AllSeeds.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\n DONE.")

##**Clinical Data**

In [ ]:
'''
!/usr/bin/env python
 coding: utf-8
 HUMAN SUBJECTS (Binary K vs R) — Unified Group-aware LOOCV + Top-N + Tuning

 TASK
 - Binary classification:
     K vs R

 DATASET DESIGN
 - These selected chips represent one chip per swab
 - LOOCV grouping is SWAB-based using Subject + SwabID

1) Baseline LOOCV on all features
2) Top-N scan using GroupKFold over N = 1..360 (no skipping)
3) LOOCV on best Top-N feature subset
 4) GroupKFold grid search on Top-N features
5) LOOCV on tuned model
6) Fit final tuned model on all data and save artifact

PRIMARY MODEL-SELECTION METRIC
- Accuracy

ALSO RECORDED
 Accuracy
 Macro F1
 Positive-class F1 (R = positive class if LabelEncoder orders K=0, R=1)
'''

import os
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    ParameterGrid,
    cross_val_predict,
    cross_val_score,
    learning_curve,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")


FORCE_RERUN = False
GK_N_SPLITS = 5
SEED_LIST = list(range(1, 1))
TOPN_MAX = 360

TASK_NAME = "Human Subjects Binary: K vs R"
TASK_TYPE = "binary"
PRIMARY_SCORING = "accuracy"

BASE_FOLDER = Path("/groups/jyyoon/OneHealth Skin Microbiome/Human Subject Data")

DATA_FILE = BASE_FOLDER / "Raw_Flow_7Lanes" / "ML_Aligned_Complete_Pivot_Table_AllSubjects_Raw_Average_aic_45s_1vps_031526.csv"

RESULTS_DIR = BASE_FOLDER / "All_Subjects_Results" / "ML_Results_AllSubjects_Unified_1ChipPerSwab_FixedChipList_360TopN_031526"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Keep exactly one chip per swab using these chips
SELECTED_CHIPS = [1, 4, 7, 10, 13, 16, 19, 22, 25, 28]

# Use subject-specific swab grouping
MERGE_SWABS_ACROSS_SUBJECTS = False
USE_SWAB_LOOCV = True

MODEL_CMAPS = {
    "Random Forest": plt.get_cmap("Purples"),
    "SVM": plt.get_cmap("Greens"),
    "Logistic Regression": plt.get_cmap("Oranges"),
    "LDA": plt.get_cmap("YlOrBr"),
    "XGBoost": plt.get_cmap("Blues"),
}


GRID = {
    "Random Forest": {
        "n_estimators": [200, 400, 800],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced"],
    },

    "SVM": [
        {
            "kernel": ["rbf"],
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", 0.01, 0.001],
            "class_weight": [None, "balanced"],
        },
        {
            "kernel": ["linear"],
            "C": [0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
        },
    ],

    "Logistic Regression": [
        {
            "solver": ["lbfgs"],
            "penalty": ["l2"],
            "C": [0.01, 0.1, 1, 10, 100],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
        },
        {
            "solver": ["saga"],
            "penalty": ["l1", "l2", "elasticnet"],
            "C": [0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0.0, 0.5, 1.0],
            "class_weight": [None, "balanced"],
            "max_iter": [2000],
        },
    ],

    "LDA": [
        {"solver": ["svd"]},
        {"solver": ["lsqr", "eigen"], "shrinkage": ["auto", 0.1, 0.5]},
    ],
}

if HAS_XGB:
    GRID["XGBoost"] = {
        "n_estimators": [100, 200, 400],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }


def make_models_for_seed(seed: int):
    models = {
        "Random Forest": RandomForestClassifier(random_state=seed),
        "SVM": SVC(kernel="rbf", probability=True, random_state=seed),
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=seed
        ),
        "LDA": LinearDiscriminantAnalysis(),
    }
    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            eval_metric="mlogloss",
            random_state=seed
        )
    return models

def add_text_page(pdf, title, text, fontsize=9):
    fig = plt.figure(figsize=(8.5, 11))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    fig.text(0.5, 0.97, title, ha="center", va="top", fontsize=11, weight="bold")
    fig.text(0.02, 0.93, text, ha="left", va="top", fontsize=fontsize, family="monospace")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

def save_metrics_row_csv(row_dict, out_csv):
    out_csv = Path(out_csv)
    df = pd.DataFrame([row_dict])
    df.to_csv(out_csv, mode="a", index=False, header=not out_csv.exists())

def plot_learning_curve_cv(estimator, X, y, groups, model_name, stage, pdf,
                           cv=None, n_points=10, rng_seed=42):
    if cv is None:
        cv = GroupKFold(n_splits=GK_N_SPLITS)

    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(estimator))])

    fig, ax = plt.subplots(figsize=(7, 5))
    train_sizes, train_scores, test_scores = learning_curve(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, n_points),
        shuffle=True,
        random_state=rng_seed,
    )
    tr_mean, tr_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
    te_mean, te_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)

    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)
    ax.fill_between(train_sizes, te_mean - te_std, te_mean + te_std, alpha=0.1)
    ax.plot(train_sizes, tr_mean, "o-", label="Training score")
    ax.plot(train_sizes, te_mean, "o-", label="CV score")
    ax.set_title(f"{model_name} — {stage} Learning Curve")
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Accuracy")
    ax.grid(True)
    ax.legend(loc="best")

    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_curve(scores_by_n, model_name, metric_name, pdf):
    fig = plt.figure()
    n_vals, sc_vals = zip(*scores_by_n)
    plt.plot(n_vals, sc_vals, marker="o")
    plt.title(f"{model_name} - {metric_name} vs Top N Features (GroupKFold)")
    plt.xlabel("Top N Features")
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def plot_feature_importance(feat_df, model_name, pdf, top_n=20):
    feat_df_sorted = feat_df.sort_values("Importance", ascending=False).head(top_n)
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_df_sorted, x="Importance", y="Feature")
    plt.title(f"{model_name} - Top {top_n} Feature Importances")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.show()
    plt.close(fig)

def _clean_intlike_to_str(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return s.astype("Int64").astype(str)

def _to_int(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def _apply_column_aliases(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}

    if "Channel" not in df.columns:
        for cand in ["ChannelNum", "Channel_Number", "ChannelID", "Channel_ID"]:
            if cand in df.columns:
                rename_map[cand] = "Channel"
                break

    if "Lane" not in df.columns:
        for cand in ["LaneNum", "Lane_Number"]:
            if cand in df.columns:
                rename_map[cand] = "Lane"
                break

    if "ChipNumber" not in df.columns:
        for cand in ["Chip", "ChipNum", "Chip_Number"]:
            if cand in df.columns:
                rename_map[cand] = "ChipNumber"
                break

    if "Subject" not in df.columns:
        for cand in ["SubjectID", "Subject_ID", "Participant", "ParticipantID"]:
            if cand in df.columns:
                rename_map[cand] = "Subject"
                break

    return df.rename(columns=rename_map)

def map_selected_chip_to_swab(chip_number: int) -> str:
    """
    Fixed one-chip-per-swab mapping:
      1  -> Swab1
      4  -> Swab2
      7  -> Swab3
      ...
      28 -> Swab10
    """
    chip_to_swab = {chip: f"Swab{i+1}" for i, chip in enumerate(SELECTED_CHIPS)}
    return chip_to_swab.get(chip_number, "SwabUnknown")

def load_and_prepare_dataframe(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = _apply_column_aliases(df)

    required = ["Bacteria", "ChipNumber", "Channel"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Available columns (first 80): {list(df.columns)[:80]}"
        )

    if "Subject" not in df.columns:
        raise ValueError(
            "Subject column is required for all-subject modeling but was not found.\n"
            "Please make sure the combined all-subjects pivot table includes a Subject column."
        )

    df["Bacteria"] = df["Bacteria"].astype(str).str.strip()
    df["Subject"] = df["Subject"].astype(str).str.strip().str.replace(r"\s+", "_", regex=True)

    df["ChipNumber"] = _to_int(df["ChipNumber"])
    df["Channel"] = _clean_intlike_to_str(df["Channel"])

    if "Lane" in df.columns:
        try:
            df["Lane"] = _clean_intlike_to_str(df["Lane"])
        except Exception:
            df["Lane"] = df["Lane"].astype(str)

    df = df[df["Bacteria"].isin(["K", "R"])].copy()
    df["BinaryLabel"] = df["Bacteria"]
    df["Combo"] = df["Bacteria"].astype(str)

    df = df.dropna(subset=["ChipNumber"]).copy()
    df["ChipNumber"] = df["ChipNumber"].astype(int)

    df = df[df["ChipNumber"].isin(SELECTED_CHIPS)].copy()

    if df.empty:
        raise RuntimeError(
            f"No rows remain after filtering to selected chips: {SELECTED_CHIPS}\n"
            f"Check DATA_FILE and chip numbering."
        )

    df["SwabID"] = df["ChipNumber"].apply(map_selected_chip_to_swab)

    bad_swabs = (df["SwabID"] == "SwabUnknown").sum()
    if bad_swabs > 0:
        raise RuntimeError(f"Found {bad_swabs} rows with unknown swab mapping after chip filtering.")

    df["GroupLabel"] = (
        df["Combo"]
        + "_Chip" + df["ChipNumber"].astype(str)
        + "_Chan" + df["Channel"].astype(str)
    )

    # Subject-specific swab grouping
    if MERGE_SWABS_ACROSS_SUBJECTS:
        df["SwabGroupLabel"] = df["SwabID"]
    else:
        df["SwabGroupLabel"] = df["Subject"] + "_" + df["SwabID"]

    print("\n Fixed one-chip-per-swab selection active")
    print("Selected chips:", SELECTED_CHIPS)
    print("Rows kept:", len(df))
    print("Unique subjects:", sorted(df["Subject"].dropna().unique().tolist()))
    print("Unique chips kept:", sorted(df["ChipNumber"].dropna().unique().tolist()))
    print("Unique swab groups:", df["SwabGroupLabel"].nunique())

    return df


DROP_COLS = [
    "Bacteria", "BinaryLabel", "Combo",
    "GroupLabel", "SwabID", "SwabGroupLabel",
    "Subject", "SubjectID", "Subject_ID", "Participant", "ParticipantID",
    "ChipNumber", "Chip", "ChipNum", "Chip_Number",
    "Channel", "ChannelNum", "Channel_Number", "ChannelID", "Channel_ID",
    "Lane", "LaneNum", "Lane_Number",
    "Concentration", "ConcentrationNum",
]

def make_X(df: pd.DataFrame) -> pd.DataFrame:
    X = (
        df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return X

def rank_features_generic(model, X, y, rng_seed):
    try:
        m = clone(model)
        if hasattr(m, "random_state"):
            try:
                m.set_params(random_state=rng_seed)
            except Exception:
                pass
        m.fit(X, y)

        if hasattr(m, "feature_importances_"):
            imp = np.asarray(m.feature_importances_)
        elif hasattr(m, "coef_"):
            coef = np.asarray(m.coef_)
            if coef.ndim == 1:
                imp = np.abs(coef)
            else:
                imp = np.abs(coef).mean(axis=0)
        else:
            raise AttributeError("No feature importance or coefficients.")
    except Exception:
        lr = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=rng_seed)
        lr.fit(X, y)
        coef = np.asarray(lr.coef_)
        if coef.ndim == 1:
            imp = np.abs(coef)
        else:
            imp = np.abs(coef).mean(axis=0)

    imp = np.ravel(imp)[:X.shape[1]]

    return (
        pd.DataFrame({"Feature": X.columns, "Importance": imp})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )


def compute_metrics(y_true, y_pred, task_type="binary"):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "MacroF1": f1_score(y_true, y_pred, average="macro"),
    }
    if task_type == "binary":
        metrics["PositiveClassF1"] = f1_score(y_true, y_pred, pos_label=1)
    return metrics

def resumable_topn_scan(
    model_name,
    base_model,
    X,
    y,
    groups,
    sorted_features,
    gk_splits,
    results_dir,
    rng_seed,
    primary_scoring="accuracy",
    task_type="binary",
):
    progress_csv = Path(results_dir) / f"TopFeatures_{model_name}_Train.csv"
    rows = []
    tried_n = set()

    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if not df_done.empty and "Top_N" in df_done.columns:
            rows = df_done.to_dict(orient="records")
            tried_n = set(df_done["Top_N"].astype(int).tolist())
            print(f"🔄 Resuming Top-N scan for {model_name}: {len(tried_n)} already done.")

    gkf = GroupKFold(n_splits=gk_splits)
    max_n = min(TOPN_MAX, len(sorted_features), X.shape[1])

    for n in range(1, max_n + 1):
        if n in tried_n:
            continue

        top_feats = sorted_features[:n]
        pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(base_model))])

        if hasattr(pipe.named_steps["est"], "random_state"):
            try:
                pipe.named_steps["est"].set_params(random_state=rng_seed)
            except Exception:
                pass

        preds = cross_val_predict(pipe, X[top_feats], y, cv=gkf, groups=groups, n_jobs=-1)
        metric_dict = compute_metrics(y, preds, task_type=task_type)

        primary_value = metric_dict["Accuracy"] if primary_scoring == "accuracy" else metric_dict["MacroF1"]

        row = {
            "Top_N": int(n),
            "Primary": float(primary_value),
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
            "PositiveClassF1": float(metric_dict["PositiveClassF1"]),
        }

        rows.append(row)
        pd.DataFrame(rows).to_csv(progress_csv, index=False)

        print(
            f"[{model_name}] Top {n} -> primary={row['Primary']:.4f} | "
            f"acc={row['Accuracy']:.4f} | macroF1={row['MacroF1']:.4f} | "
            f"posF1={row['PositiveClassF1']:.4f}"
        )

    return pd.DataFrame(rows)

def resumable_group_grid_search(
    model_name,
    base_estimator,
    param_grid,
    X,
    y,
    groups,
    results_dir,
    scoring="accuracy",
    cv_splits=5,
    random_state=42,
    tag="",
):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    progress_csv = results_dir / f"GridSearch_Resume_{model_name}{tag}.csv"
    best_params_json = results_dir / f"BestParams_{model_name}{tag}.json"

    combos = list(ParameterGrid(param_grid)) if param_grid else [{}]
    gkf = GroupKFold(n_splits=cv_splits)

    tried = set()
    if progress_csv.exists():
        df_done = pd.read_csv(progress_csv)
        if "params_json" in df_done.columns:
            tried = set(df_done["params_json"].astype(str).tolist())

    best_score, best_params = -np.inf, None
    if best_params_json.exists():
        try:
            prev = json.load(open(best_params_json))
            best_params = prev.get("best_params")
            best_score = prev.get("best_score", -np.inf)
        except Exception:
            pass

    print(f"🔎 {model_name}: {len(combos)} combos total; {len(tried)} already done.")

    for i, params in enumerate(combos, 1):
        key = json.dumps(params, sort_keys=True)
        if key in tried:
            continue

        est = clone(base_estimator)
        if hasattr(est, "random_state"):
            try:
                est.set_params(random_state=random_state)
            except Exception:
                pass

        try:
            est.set_params(**params)
        except ValueError as e:
            pd.DataFrame([{
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model": model_name,
                "params_json": key,
                "mean_score": np.nan,
                "std_score": np.nan,
                "n_folds": 0,
                "note": f"incompatible: {e}",
            }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())
            continue

        pipe = Pipeline([("scaler", StandardScaler()), ("est", est)])
        scores = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring=scoring, n_jobs=-1)
        mean_score, std_score = float(np.mean(scores)), float(np.std(scores))

        pd.DataFrame([{
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "model": model_name,
            "params_json": key,
            "mean_score": mean_score,
            "std_score": std_score,
            "n_folds": len(scores),
        }]).to_csv(progress_csv, mode="a", index=False, header=not progress_csv.exists())

        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            json.dump({"best_params": best_params, "best_score": best_score}, open(best_params_json, "w"))

        print(
            f"[{model_name}] {i}/{len(combos)} params={params} -> "
            f"{scoring}={mean_score:.4f} (±{std_score:.4f}) | best={best_score:.4f}"
        )

    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("est", clone(base_estimator).set_params(**(best_params or {})))
    ])

    return best_pipe, best_params, best_score

def evaluate_stage_loocv(
    model,
    X,
    y,
    groups,
    model_name,
    stage,
    cmap,
    labels,
    pdf,
    results_dir,
    seed,
    data_file_name,
    task_type="binary",
):
    logo = LeaveOneGroupOut()
    pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(model))])

    if hasattr(pipe.named_steps["est"], "random_state"):
        try:
            pipe.named_steps["est"].set_params(random_state=seed)
        except Exception:
            pass

    preds = cross_val_predict(pipe, X, y, cv=logo, groups=groups, n_jobs=-1)
    metric_dict = compute_metrics(y, preds, task_type=task_type)

    report_str = classification_report(y, preds, target_names=labels)

    print(
        f"\n{model_name} - {stage} LOOCV "
        f"| Acc={metric_dict['Accuracy']:.3f} "
        f"| MacroF1={metric_dict['MacroF1']:.3f} "
        f"| PosF1={metric_dict['PositiveClassF1']:.3f}"
    )

    add_text_page(pdf, f"{model_name} — {stage} LOOCV (seed {seed})", report_str)

    fig_cm = plt.figure()
    sns.heatmap(confusion_matrix(y, preds), annot=True, fmt="d",
                cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(f"{model_name} - {stage} Confusion Matrix (LOOCV, seed {seed})")
    plt.tight_layout()
    pdf.savefig(fig_cm)
    plt.show()
    plt.close(fig_cm)

    try:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="predict_proba", n_jobs=-1)[:, 1]
    except Exception:
        y_scores = cross_val_predict(pipe, X, y, cv=logo, groups=groups,
                                     method="decision_function", n_jobs=-1)
        y_scores = np.asarray(y_scores).ravel()

    fpr, tpr, _ = roc_curve(y, y_scores)
    prec, rec, _ = precision_recall_curve(y, y_scores)

    fig_roc, ax_roc = plt.subplots()
    ax_roc.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.2f}")
    ax_roc.plot([0, 1], [0, 1], "k--")
    ax_roc.set_title(f"{model_name} - {stage} ROC (LOOCV, seed {seed})")
    ax_roc.set_xlabel("FPR")
    ax_roc.set_ylabel("TPR")
    ax_roc.legend()
    plt.tight_layout()
    pdf.savefig(fig_roc)
    plt.show()
    plt.close(fig_roc)

    fig_pr, ax_pr = plt.subplots()
    ax_pr.plot(rec, prec, label=f"AUC={auc(rec, prec):.2f}")
    ax_pr.set_title(f"{model_name} - {stage} PR (LOOCV, seed {seed})")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend()
    plt.tight_layout()
    pdf.savefig(fig_pr)
    plt.show()
    plt.close(fig_pr)

    save_metrics_row_csv(
        {
            "Model": model_name,
            "Stage": stage,
            "Seed": seed,
            "Accuracy": float(metric_dict["Accuracy"]),
            "MacroF1": float(metric_dict["MacroF1"]),
            "PositiveClassF1": float(metric_dict["PositiveClassF1"]),
            "DataFile": data_file_name,
        },
        Path(results_dir) / "All_Metrics_LOOCV.csv"
    )

    return metric_dict, preds


def print_group_summary(X, y, groups, labels=None, dataset_name="Dataset"):
    import pandas as pd
    import numpy as np

    groups_series = pd.Series(groups).astype(str)
    y_array = np.asarray(y)

    print("\n" + "=" * 70)
    print(f"{dataset_name} SUMMARY")
    print("=" * 70)
    print(f"Rows: {len(y_array):,}")
    print(f"Features: {X.shape[1]:,}")
    print(f"Unique groups: {groups_series.nunique():,}")

    group_sizes = groups_series.value_counts()
    print(f"Min rows per group: {group_sizes.min():,}")
    print(f"Max rows per group: {group_sizes.max():,}")
    print(f"Mean rows per group: {group_sizes.mean():.2f}")

    if labels is not None:
        print(f"Classes ({len(labels)}): {labels}")

    print("\nFirst 10 groups:")
    print(group_sizes.head(10).to_string())

    print("=" * 70 + "\n")


def evaluate_model_end_to_end(
    model_name,
    base_model,
    X,
    y,
    groups,
    labels,
    cmap,
    gk_splits,
    rng_seed,
    results_dir_seed,
    data_file_name,
    primary_scoring="accuracy",
    task_type="binary",
):
    pdf_path = Path(results_dir_seed) / f"{model_name}_Results_Seed{rng_seed}.pdf"
    summary = {
        "Model": model_name,
        "Seed": rng_seed,
        "Task": TASK_NAME,
        "TaskType": task_type,
        "PrimaryScoring": primary_scoring,
    }

    gkf_for_curves = GroupKFold(n_splits=gk_splits)

    with PdfPages(pdf_path) as pdf:

        baseline_metrics, _ = evaluate_stage_loocv(
            base_model, X, y, groups,
            model_name, "Baseline", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Baseline_LOOCV_Accuracy"] = baseline_metrics["Accuracy"]
        summary["Baseline_LOOCV_MacroF1"] = baseline_metrics["MacroF1"]
        summary["Baseline_LOOCV_PositiveClassF1"] = baseline_metrics["PositiveClassF1"]

        plot_learning_curve_cv(
            base_model, X, y, groups, model_name, "Baseline", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        feat_df = rank_features_generic(base_model, X, y, rng_seed)
        sorted_names = feat_df["Feature"].tolist()[:min(TOPN_MAX, X.shape[1])]

        df_topn = resumable_topn_scan(
            model_name=model_name,
            base_model=base_model,
            X=X,
            y=y,
            groups=groups,
            sorted_features=sorted_names,
            gk_splits=gk_splits,
            results_dir=results_dir_seed,
            rng_seed=rng_seed,
            primary_scoring=primary_scoring,
            task_type=task_type,
        )

        best_row = df_topn.sort_values("Primary", ascending=False).iloc[0]
        best_n = int(best_row["Top_N"])
        top_features = sorted_names[:best_n]
        summary["TopN"] = best_n

        plot_feature_curve(
            list(zip(df_topn["Top_N"].tolist(), df_topn["Primary"].tolist())),
            model_name,
            f"Primary({primary_scoring})",
            pdf
        )
        plot_feature_importance(feat_df, model_name, pdf)

        pd.DataFrame({"Feature": top_features}).to_csv(
            Path(results_dir_seed) / f"TopFeatures_{model_name}_Selected.csv",
            index=False
        )

        top_metrics, _ = evaluate_stage_loocv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["TopFeatures_LOOCV_Accuracy"] = top_metrics["Accuracy"]
        summary["TopFeatures_LOOCV_MacroF1"] = top_metrics["MacroF1"]
        summary["TopFeatures_LOOCV_PositiveClassF1"] = top_metrics["PositiveClassF1"]

        plot_learning_curve_cv(
            base_model, X[top_features], y, groups,
            model_name, f"Top {best_n} Features", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        _, best_params, _ = resumable_group_grid_search(
            model_name=model_name,
            base_estimator=base_model,
            param_grid=GRID.get(model_name, {}),
            X=X[top_features],
            y=y,
            groups=groups,
            results_dir=results_dir_seed,
            scoring=primary_scoring,
            cv_splits=gk_splits,
            random_state=rng_seed,
        )

        tuned_best = clone(base_model).set_params(**(best_params or {}))
        if hasattr(tuned_best, "random_state"):
            try:
                tuned_best.set_params(random_state=rng_seed)
            except Exception:
                pass

        tuned_metrics, _ = evaluate_stage_loocv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", cmap, labels, pdf,
            results_dir_seed, rng_seed, data_file_name,
            task_type=task_type
        )
        summary["Tuned_LOOCV_Accuracy"] = tuned_metrics["Accuracy"]
        summary["Tuned_LOOCV_MacroF1"] = tuned_metrics["MacroF1"]
        summary["Tuned_LOOCV_PositiveClassF1"] = tuned_metrics["PositiveClassF1"]
        summary["Best_Params"] = best_params

        plot_learning_curve_cv(
            tuned_best, X[top_features], y, groups,
            model_name, "Tuned", pdf,
            cv=gkf_for_curves, rng_seed=rng_seed
        )

        final_pipe = Pipeline([("scaler", StandardScaler()), ("est", clone(tuned_best))])
        final_pipe.fit(X[top_features], y)

        artifact = {
            "pipeline": final_pipe,
            "top_features": top_features,
            "labels": labels,
            "best_params": best_params,
            "data_file": data_file_name,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "seed": rng_seed,
            "task": TASK_NAME,
            "primary_scoring": primary_scoring,
            "task_type": task_type,
            "selected_chips": SELECTED_CHIPS,
        }
        joblib.dump(artifact, Path(results_dir_seed) / f"{model_name}_Final_Fitted.joblib")

    return summary

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    df_all = load_and_prepare_dataframe(DATA_FILE)

    X = make_X(df_all)
    if X.shape[1] == 0:
        raise RuntimeError("No numeric feature columns found after dropping metadata columns.")

    le = LabelEncoder()
    y = le.fit_transform(df_all["BinaryLabel"])
    labels = list(le.classes_)

    if USE_SWAB_LOOCV:
        groups = df_all["SwabGroupLabel"].astype(str).values
        print(f"\nUsing SWAB-based LOOCV groups. Groups={pd.Series(groups).nunique()}")
    else:
        groups = df_all["GroupLabel"].astype(str).values
        print(f"\nUsing original combo-chip-channel LOOCV groups. Groups={pd.Series(groups).nunique()}")

    print(f" DATA_FILE = {DATA_FILE}")
    print(f" Rows = {len(df_all):,}")
    print(f" Subjects = {sorted(df_all['Subject'].dropna().unique().tolist())}")
    print(f" Features = {X.shape[1]}")
    print(f" Labels = {labels}")
    print(f" Primary scoring = {PRIMARY_SCORING}")
    print(f" XGBoost available = {HAS_XGB}")

    all_summaries = []

    for seed in SEED_LIST:
        print(f"\n SEED {seed} ")
        results_dir_seed = RESULTS_DIR / f"Seed_{seed}"
        results_dir_seed.mkdir(parents=True, exist_ok=True)

        MODELS = make_models_for_seed(seed)
        per_seed_summaries = []

        for model_name, model in MODELS.items():
            print(f"\n🚀 Seed {seed} — Running {model_name}")

            cmap = MODEL_CMAPS.get(model_name, plt.get_cmap("Blues"))

            summary = evaluate_model_end_to_end(
                model_name=model_name,
                base_model=model,
                X=X,
                y=y,
                groups=groups,
                labels=labels,
                cmap=cmap,
                gk_splits=GK_N_SPLITS,
                rng_seed=seed,
                results_dir_seed=results_dir_seed,
                data_file_name=Path(DATA_FILE).name,
                primary_scoring=PRIMARY_SCORING,
                task_type=TASK_TYPE,
            )
            per_seed_summaries.append(summary)
            all_summaries.append(summary)

        per_seed_df = pd.DataFrame(per_seed_summaries)
        col_order = [
            "Model",
            "Seed",
            "Task",
            "TaskType",
            "PrimaryScoring",
            "Baseline_LOOCV_Accuracy",
            "Baseline_LOOCV_MacroF1",
            "Baseline_LOOCV_PositiveClassF1",
            "TopN",
            "TopFeatures_LOOCV_Accuracy",
            "TopFeatures_LOOCV_MacroF1",
            "TopFeatures_LOOCV_PositiveClassF1",
            "Tuned_LOOCV_Accuracy",
            "Tuned_LOOCV_MacroF1",
            "Tuned_LOOCV_PositiveClassF1",
            "Best_Params",
        ]
        cols_present = [c for c in col_order if c in per_seed_df.columns]
        per_seed_df = per_seed_df.loc[:, cols_present + [c for c in per_seed_df.columns if c not in cols_present]]

        per_seed_path = results_dir_seed / "Model_Comparison_Summary_Binary_Train.csv"
        per_seed_df.to_csv(per_seed_path, index=False)
        print(f" Seed {seed} summary saved → {per_seed_path}")

    summary_df = pd.DataFrame(all_summaries)
    col_order = [
        "Model",
        "Seed",
        "Task",
        "TaskType",
        "PrimaryScoring",
        "Baseline_LOOCV_Accuracy",
        "Baseline_LOOCV_MacroF1",
        "Baseline_LOOCV_PositiveClassF1",
        "TopN",
        "TopFeatures_LOOCV_Accuracy",
        "TopFeatures_LOOCV_MacroF1",
        "TopFeatures_LOOCV_PositiveClassF1",
        "Tuned_LOOCV_Accuracy",
        "Tuned_LOOCV_MacroF1",
        "Tuned_LOOCV_PositiveClassF1",
        "Best_Params",
    ]
    cols_present = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df.loc[:, cols_present + [c for c in summary_df.columns if c not in cols_present]]

    summary_path = RESULTS_DIR / "Model_Comparison_Summary_Binary_AllSeeds.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\n DONE.")